# QIREN — ICML 2024 Table 1 reproduction (paper constraints + official implementation)

Paper: [Quantum Implicit Neural Representations](https://arxiv.org/pdf/2406.03873) · Official code: [GGorMM1/QIREN](https://github.com/GGorMM1/QIREN)  
**Scope**: signal representation for Sound / Astronaut / Camera / Coffee and 32→64 image super-resolution from Table 1; the paper's GAN image-generation experiment is **not** included.

### Specified explicitly in Appendix C.1 of the paper
- MSE loss; Adam with β₁=0.9, β₂=0.999, ε=1e-8.
- Astronaut **600** epochs; Sound, Camera and Coffee **300** epochs each.
- Each model is run 5 times independently and the best result is reported. ReLU/Tanh/RFF/SIREN: hidden=10, hidden_layers=6; QIREN: 3 hybrid layers, 8 qubits, width 8 per layer.
- Images from `skimage.data`, trained at 32×32, with the 64×64 image as the super-resolution ground truth; Sound uses the Bach wav from the original repository.

### Not published or not fully specified in the paper, so this notebook **cannot guarantee a digit-for-digit reproduction**
1. Exact learning rate per task, SIREN's best ω₀ and the hyperparameter search range. Other models use the **official train.py default lr=0.03**; the SIREN search is recorded separately (**these are not values published by the authors**).
2. The official trainer calls `model.train()` every epoch, and the official example does not switch to `eval()` for the forward pass after training. The paper does not state the final BatchNorm inference protocol. **We report both modes**, clearly separated.
3. The authors' seeds for the five random initializations, the rule for choosing the best checkpoint, and the Python/PyTorch/PennyLane versions are not fully given. Here the five seeds are fixed to 0–4, and the commit hash and runtime versions are saved.

**Safety**: results go to a new output folder `MyDrive/QIREN_paper_recheck` and never overwrite earlier experiments. Weights and logs are saved for every finished seed; after a Colab disconnect, training resumes from the unfinished seed, **not from the interrupted epoch**.

**How to run**: run cells 1–4, then training cell 5, then evaluation cells 6–7. SIREN tuning requires additional full training runs (keep the default settings to include it).

In [ ]:
# ===== Cell 1: clean official checkout, Google Drive, environment =====
import os, sys, subprocess, json, pathlib, importlib.metadata, platform
from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

ROOT = pathlib.Path('/content/QIREN_paper_recheck_repo')
QINR = ROOT / 'qinr'
OUT = pathlib.Path('/content/drive/MyDrive/QIREN_paper_recheck')
OUT.mkdir(parents=True, exist_ok=True)

if not (ROOT / '.git').is_dir():
    assert not ROOT.exists(), f'{ROOT} exists but is not a clean Git checkout; rename/remove it first.'
    subprocess.run(['git', 'clone', 'https://github.com/GGorMM1/QIREN.git', str(ROOT)], check=True)
REPO_SHA = subprocess.check_output(['git', '-C', str(ROOT), 'rev-parse', 'HEAD'], text=True).strip()
print('Official repository:', ROOT)
print('Commit SHA:', REPO_SHA)

# Use Colab's already-installed PyTorch / torchvision; install PennyLane only.
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'pennylane'], check=True)
import torch, torchvision, pennylane as qml, skimage, scipy
manifest = {'repo': 'https://github.com/GGorMM1/QIREN', 'commit': REPO_SHA,
            'python': platform.python_version(), 'torch': torch.__version__,
            'torchvision': torchvision.__version__, 'pennylane': qml.__version__,
            'skimage': skimage.__version__, 'scipy': scipy.__version__}
manifest_path = OUT / 'environment.json'
if manifest_path.exists():
    old = json.loads(manifest_path.read_text())
    assert old['commit'] == REPO_SHA, 'Repository commit changed. Do not mix results from different versions.'
else:
    manifest_path.write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))

Mounted at /content/drive
Official repository: /content/QIREN_paper_recheck_repo
Commit SHA: 56ab4010ee3fbb3371c9d5d5f5284a8c41d4e6f3
{
  "repo": "https://github.com/GGorMM1/QIREN",
  "commit": "56ab4010ee3fbb3371c9d5d5f5284a8c41d4e6f3",
  "python": "3.13.18",
  "torch": "2.14.1+cpu",
  "torchvision": "0.29.1+cpu",
  "pennylane": "0.45.1",
  "skimage": "0.25.2",
  "scipy": "1.16.3"
}


## Compatibility patches only (network architecture, optimization and image transforms unchanged)

The official repository only loads Coffee by default and saves every epoch to `./checkpoint/`. Below we add image selection, a seed and a separate checkpoint path, and comment out an unused legacy Qiskit import. The trainer's forward pass, backward pass, optimizer and loss are unchanged.

In [ ]:
# ===== Cell 2: apply minimal, checked patches to the official checkout =====
from pathlib import Path

def checked_patch(relpath, replacements):
    p = QINR / relpath
    text = p.read_text(encoding='utf-8').replace('\r\n', '\n')
    if text.startswith('#QIREN-PAPER-RECHECK-PATCHED'):
        print('Already patched:', relpath); return
    for before, after in replacements:
        count = text.count(before)
        if count != 1:
            raise RuntimeError(f'{relpath}: required patch anchor occurs {count} times: {before!r}')
        text = text.replace(before, after, 1)
    p.write_text('#QIREN-PAPER-RECHECK-PATCHED\n' + text, encoding='utf-8')
    print('Patched:', relpath)

checked_patch('modules.py', [
    ('import qiskit.providers.aer.noise as noise',
     'try:\n    import qiskit.providers.aer.noise as noise\nexcept ImportError:\n    noise = None  # unused in QIREN default noiseless simulation'),
])
checked_patch('image/data_process.py', [
    ('def get_cameraman_tensor(sidelength):',
     "def get_cameraman_tensor(sidelength, name='coffee'):"),
    ('    img = Image.fromarray(skimage.color.rgb2gray(skimage.data.coffee()))\n',
     "    if name == 'camera':\n"
     "        img = Image.fromarray(skimage.data.camera())\n"
     "    elif name == 'astronaut':\n"
     "        img = Image.fromarray(skimage.color.rgb2gray(skimage.data.astronaut()))\n"
     "    elif name == 'coffee':\n"
     "        img = Image.fromarray(skimage.color.rgb2gray(skimage.data.coffee()))\n"
     "    else:\n"
     "        raise ValueError(f'Unknown image: {name}')\n"),
    ('    def __init__(self, sidelength):', "    def __init__(self, sidelength, name='coffee'):"),
    ('        img = get_cameraman_tensor(sidelength)',
     '        img = get_cameraman_tensor(sidelength, name)'),
])
checked_patch('train.py', [
    ("parser.add_argument('--use_noise', type=float, default=0)\n",
     "parser.add_argument('--use_noise', type=float, default=0)\n"
     "parser.add_argument('--image', type=str, choices=['coffee', 'camera', 'astronaut'], default='coffee')\n"
     "parser.add_argument('--ckpt_dir', type=str, default='./checkpoint')\n"
     "parser.add_argument('--seed', type=int, default=-1)\n"),
    ('args = parser.parse_args()\n',
     'args = parser.parse_args()\n'
     'if args.seed >= 0:\n'
     '    import random\n'
     '    random.seed(args.seed)\n'
     '    np.random.seed(args.seed)\n'
     '    torch.manual_seed(args.seed)\n'),
    ('path = "./checkpoint/" + str(epoch) + ".pth"',
     'os.makedirs(args.ckpt_dir, exist_ok=True); path = os.path.join(args.ckpt_dir, str(epoch) + ".pth")'),
    ('ImageFitting(args.img_size)', 'ImageFitting(args.img_size, args.image)'),
])

subprocess.run([sys.executable, '-m', 'py_compile', 'train.py', 'modules.py', 'image/data_process.py'], cwd=QINR, check=True)
print('All three files compile. Models, training loop, and data transforms remain official.')

Patched: modules.py
Patched: image/data_process.py
Patched: train.py
All three files compile. Models, training loop, and data transforms remain official.


## Pre-check: task data, parameter counts and the quantum network

Parameter counts must match Table 1 exactly (Sound/Image: QIREN 649/657; SIREN 691/701; ReLU/Tanh 831/841; RFF 791/791). Any mismatch raises an error.

In [ ]:
# ===== Cell 3: preflight — exact model counts and dataset difference =====
import os, sys, torch, numpy as np, hashlib
os.chdir(QINR)
if str(QINR) not in sys.path: sys.path.insert(0, str(QINR))
from modules import Hybridren, Siren, Siren_bn
from image.data_process import ImageFitting
from sound.data_process import AudioFile

SEEDS = (0, 1, 2, 3, 4)
PAPER_EPOCHS = {'sound':300, 'coffee':300, 'camera':300, 'astronaut':600}
EXPECTED = {'relu':(831,841), 'tanh':(831,841), 'relu+rff':(791,791),
            'siren':(691,701), 'hybridren':(649,657)}
OMEGA_GRID = (1, 10, 30)
LR_GRID = (1e-4, 1e-3, 1e-2, 3e-2)
TUNE_SIREN = True  # the paper says SIREN is tuned, but does NOT disclose its exact grid

# Model builder is a direct mirror of official train.py model branches.
def make_model(model, in_f, omega=1):
    if model == 'hybridren':
        return Hybridren(in_features=in_f, out_features=1, hidden_features=8,
                         hidden_layers=2, spectrum_layer=2, use_noise=0)
    if model == 'siren':
        return Siren(in_features=in_f, out_features=1, hidden_features=10,
                     hidden_layers=6, first_omega_0=omega, hidden_omega_0=omega)
    activ = {'relu':'relu', 'tanh':'tanh', 'relu+rff':'relu'}[model]
    return Siren_bn(in_features=in_f, out_features=1, hidden_features=10,
                    hidden_layers=6, first_omega_0=1, hidden_omega_0=1,
                    activ=activ, rff=(model == 'relu+rff'))

for task, dim in [('sound',1),('camera',2)]:
    for model, expected in EXPECTED.items():
        n = sum(p.numel() for p in make_model(model, dim).parameters() if p.requires_grad)
        assert n == expected[0 if dim==1 else 1], (task, model, n, expected)
        print(f'OK #params {task:<6} {model:<10} {n}')

print('\nData preview (shape, mean, standard deviation, SHA256):')
for name in ('coffee','camera','astronaut'):
    coords, pixels = ImageFitting(32, name)[0]
    assert coords.shape == (1024,2) and pixels.shape == (1024,1)
    h = hashlib.sha256(pixels.numpy().tobytes()).hexdigest()[:16]
    print(name, pixels.shape, round(pixels.mean().item(),5), round(pixels.std().item(),5), h)
coords, pixels = AudioFile('./sound/gt_bach.wav')[0]
assert coords.shape == (1000,1) and pixels.shape == (1000,1)
print('sound', pixels.shape, 'range', (pixels.min().item(), pixels.max().item()))

OK #params sound  relu       831
OK #params sound  tanh       831
OK #params sound  relu+rff   791
OK #params sound  siren      691
OK #params sound  hybridren  649
OK #params camera relu       841
OK #params camera tanh       841
OK #params camera relu+rff   791
OK #params camera siren      701
OK #params camera hybridren  657

Data preview (shape, mean, standard deviation, SHA256):
coffee torch.Size([1024, 1]) -0.28148 0.44649 0428de3447dd0ac4
camera torch.Size([1024, 1]) 0.01223 0.53834 693ef2fd7322b4fb
astronaut torch.Size([1024, 1]) -0.11619 0.50534 59a17a19eb8f10a3
sound torch.Size([1000, 1]) range (-0.5609756112098694, 1.0)


/usr/local/lib/python3.13/dist-packages/torch/functional.py:504: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/native/TensorShape.cpp:4221.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


## Trainer: official `train.py`, all results saved to Drive

**Note:** the official code saves a checkpoint every epoch. This script keeps them under `/content/` and copies only the final weights to Drive after each run, to limit the number of Drive files. A disconnect only reruns the interrupted seed.

In [ ]:
# ===== Cell 4: one-run helper; exact official 300/600 epochs, not 301/601 =====
import subprocess, shutil, json, re, time, pathlib, math
RUNS = OUT / 'runs'; RUNS.mkdir(exist_ok=True)
TUNES = OUT / 'siren_tune'; TUNES.mkdir(exist_ok=True)
TEMP = pathlib.Path('/content/qiren_tmp_ckpt'); TEMP.mkdir(exist_ok=True)

TASK_ARGS = {
    'sound': ('--type','sound','--in_features','1'),
    'coffee': ('--type','image','--in_features','2','--image','coffee'),
    'camera': ('--type','image','--in_features','2','--image','camera'),
    'astronaut': ('--type','image','--in_features','2','--image','astronaut')
}

def run_one(task, model, seed, lr=0.03, omega=1, output_dir=RUNS, prefix=''):
    """Train from scratch using official train.py; resumable between complete runs."""
    E = PAPER_EPOCHS[task]
    tag = f'{prefix}{task}__{model}__seed{seed}'
    jf = output_dir / f'{tag}.json'
    wf = output_dir / f'{tag}.pth'
    if jf.exists() and wf.exists():
        rec = json.loads(jf.read_text())
        assert rec['repo_sha'] == REPO_SHA
        assert rec['epochs'] == E and rec['lr'] == lr and rec['omega'] == omega
        print('SKIP completed:', tag)
        return rec
    args = ['--model', model, '--hidden_features', str(8 if model=='hybridren' else 10),
            '--hidden_layers', str(2 if model=='hybridren' else 6),
            '--learning_rate', str(lr), '--seed', str(seed)]
    if model == 'hybridren':
        args += ['--spectrum_layer', '2', '--use_noise', '0']
    elif model == 'siren':
        args += ['--first_omega_0', str(omega), '--hidden_omega_0', str(omega)]
    ck = TEMP / tag
    if ck.exists(): shutil.rmtree(ck)  # remove debris from an interrupted run
    ck.mkdir(parents=True)
    cmd = [sys.executable, '-u', 'train.py', *TASK_ARGS[task], *args,
           '--epochs', str(E), '--ckpt_dir', str(ck)]
    logf = output_dir / f'{tag}.log'
    print('START', tag, 'lr=', lr, 'omega=', omega, flush=True)
    started = time.time()
    with open(logf, 'w', encoding='utf-8') as log:
        p = subprocess.Popen(cmd, cwd=QINR, stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
        loss_list = []; nparams = None; tail = []
        for line in p.stdout:
            log.write(line)
            tail.append(line.rstrip()); tail = tail[-35:]
            if m := re.search(r'parameters:\s*(\d+)', line): nparams = int(m.group(1))
            if m := re.search(r'epoch\s+(\d+),\s+Total loss\s+([-\deE+.]+)', line):
                ep, loss = int(m.group(1)), float(m.group(2))
                loss_list.append(loss)
                if ep % 100 == 0 or ep == E-1:
                    print(f'  {tag} epoch={ep:3} MSE×1e-3={loss*1000:.4f}', flush=True)
        rc = p.wait()
    if rc != 0 or len(loss_list) != E:
        raise RuntimeError(f'RUN FAILED {tag}: returncode={rc}, #epochs={len(loss_list)} expected={E}\n'
                           + '\n'.join(tail) + f'\nSee {logf}')
    expected = EXPECTED[model][0 if task=='sound' else 1]
    assert nparams == expected, (tag, 'param mismatch', nparams, expected)
    end_ck = ck / f'{E-1}.pth'
    assert end_ck.is_file(), f'Final checkpoint missing: {end_ck}'
    shutil.copy2(end_ck, wf)
    rec = dict(task=task, model=model, seed=seed, lr=lr, omega=omega,
               epochs=E, params=nparams, repo_sha=REPO_SHA, duration_s=round(time.time()-started,2),
               last_pre_update_loss=loss_list[-1], losses=loss_list, command=cmd,
               checkpoint=str(wf))
    tmp = jf.with_suffix('.json.tmp'); tmp.write_text(json.dumps(rec, indent=2)); os.replace(tmp,jf)
    shutil.rmtree(ck, ignore_errors=True)
    print('DONE', tag, 'last MSE×1e-3', round(loss_list[-1]*1000,4), flush=True)
    return rec

## SIREN hyperparameter search (recorded separately; not presented as the paper's published settings)

Appendix C.1 of the paper only says that SIREN uses its special initialization and tuned hyperparameters. Here we search ω₀ ∈ {1,10,30} and lr ∈ {1e-4,1e-3,1e-2,3e-2} on a separate seed (123), fix the best combination per task, and then run seeds 0–4. **The result should be called a tuned SIREN baseline, not "the authors' hyperparameters".**

In [ ]:
# ===== Cell 5: explicit SIREN tuning (not fully specified by paper) =====
import json, itertools
TUNE_SEED = 123
best_siren = {}
settings_file = OUT / 'siren_selected_hyperparams.json'
if TUNE_SIREN:
    for task in PAPER_EPOCHS:
        candidates = []
        for omega, lr in itertools.product(OMEGA_GRID, LR_GRID):
            # Unique tags prevent tuning runs from overwriting one another.
            prefix = f'omega{omega}__lr{lr:g}__'
            rec = run_one(task, 'siren', TUNE_SEED, lr=lr, omega=omega,
                          output_dir=TUNES, prefix=prefix)
            candidates.append((rec['last_pre_update_loss'], omega, lr))
        score, omega, lr = min(candidates)
        best_siren[task] = {'omega':omega, 'lr':lr, 'tune_seed':TUNE_SEED,
                            'tune_last_loss':score}
        print('TUNED SIREN', task, 'omega=', omega, 'lr=', lr,
              'tune MSE×1e-3=', score*1000)
    settings_file.write_text(json.dumps(best_siren, indent=2))
else:
    # This is OFFICIAL REPO DEFAULT, NOT paper's disclosed tuned SIREN.
    best_siren = {task:{'omega':1, 'lr':0.03, 'tune_seed':None} for task in PAPER_EPOCHS}
    settings_file.write_text(json.dumps(best_siren, indent=2))
print('Selected hyperparameters saved to', settings_file)

START omega1__lr0.0001__sound__siren__seed123 lr= 0.0001 omega= 1
  omega1__lr0.0001__sound__siren__seed123 epoch=  0 MSE×1e-3=1857.0050
  omega1__lr0.0001__sound__siren__seed123 epoch=100 MSE×1e-3=242.0080
  omega1__lr0.0001__sound__siren__seed123 epoch=200 MSE×1e-3=44.6460
  omega1__lr0.0001__sound__siren__seed123 epoch=299 MSE×1e-3=32.5200
DONE omega1__lr0.0001__sound__siren__seed123 last MSE×1e-3 32.52
START omega1__lr0.001__sound__siren__seed123 lr= 0.001 omega= 1
  omega1__lr0.001__sound__siren__seed123 epoch=  0 MSE×1e-3=1857.0050
  omega1__lr0.001__sound__siren__seed123 epoch=100 MSE×1e-3=26.0810
  omega1__lr0.001__sound__siren__seed123 epoch=200 MSE×1e-3=23.6200
  omega1__lr0.001__sound__siren__seed123 epoch=299 MSE×1e-3=20.9460
DONE omega1__lr0.001__sound__siren__seed123 last MSE×1e-3 20.946
START omega1__lr0.01__sound__siren__seed123 lr= 0.01 omega= 1
  omega1__lr0.01__sound__siren__seed123 epoch=  0 MSE×1e-3=1857.0050
  omega1__lr0.01__sound__siren__seed123 epoch=100 MSE×1e

In [ ]:

from pathlib import Path
import sys
import importlib

p = Path("/content/QIREN_paper_recheck_repo/qinr/modules.py")
s = p.read_text()

old = "qml.RZ(inputs[j], wires=j)"
new = "qml.RZ(inputs[..., j], wires=j)"

if old in s:
    p.write_text(s.replace(old, new, 1))
    print("QuantumLayer batch indexing fixed.")
elif new in s:
    print("Already fixed.")
else:
    raise RuntimeError("Cannot locate the expected RZ statement.")

# Reload modules in the current notebook kernel
if "modules" in sys.modules:
    importlib.reload(sys.modules["modules"])

print("Patch complete.")


QuantumLayer batch indexing fixed.
Patch complete.


## Main training (five models × four tasks × five seeds)

Runs use the official architectures, MSE, Adam and the full number of epochs, with a separate output folder. **Do not reuse result files from older notebooks.** QIREN's PennyLane quantum simulation is computationally expensive.

In [ ]:
# ===== Cell 6: paper-aligned full runs =====
MODELS = ('relu', 'tanh', 'relu+rff', 'siren', 'hybridren')
# To diagnose only one model first, temporarily change to ('hybridren',)
for model in MODELS:
    for task in PAPER_EPOCHS:
        if model == 'siren':
            hp = best_siren[task]
            lr, omega = hp['lr'], hp['omega']
        else:
            lr, omega = 0.03, 1
        for seed in SEEDS:
            run_one(task, model, seed, lr=lr, omega=omega)
print('Finished all requested runs. Files in:', RUNS)

SKIP completed: sound__relu__seed0
SKIP completed: sound__relu__seed1
SKIP completed: sound__relu__seed2
SKIP completed: sound__relu__seed3
SKIP completed: sound__relu__seed4
SKIP completed: coffee__relu__seed0
SKIP completed: coffee__relu__seed1
SKIP completed: coffee__relu__seed2
SKIP completed: coffee__relu__seed3
SKIP completed: coffee__relu__seed4
SKIP completed: camera__relu__seed0
SKIP completed: camera__relu__seed1
SKIP completed: camera__relu__seed2
SKIP completed: camera__relu__seed3
SKIP completed: camera__relu__seed4
SKIP completed: astronaut__relu__seed0
SKIP completed: astronaut__relu__seed1
SKIP completed: astronaut__relu__seed2
SKIP completed: astronaut__relu__seed3
SKIP completed: astronaut__relu__seed4
SKIP completed: sound__tanh__seed0
SKIP completed: sound__tanh__seed1
SKIP completed: sound__tanh__seed2
SKIP completed: sound__tanh__seed3
SKIP completed: sound__tanh__seed4
SKIP completed: coffee__tanh__seed0
SKIP completed: coffee__tanh__seed1
SKIP completed: coffee_

## Strict evaluation: train mode and eval mode computed from the same checkpoint

- **Rep (32×32 / Sound)**: MSE from a fresh forward pass with the final weights, rather than the last printed loss.
- **SR (64×64)**: the same model trained on 32×32, queried on new 64×64 coordinates and compared with a separate 64×64 ground truth.
- **train mode**: matches the forward pass in the official post-training example; BatchNorm uses the statistics of the current coordinate batch. This is not equivalent to inference in a deployment sense.
- **eval mode**: standard PyTorch evaluation with fixed running statistics.
- **Each mode loads the checkpoint separately**, so a train-mode forward pass cannot update the BN buffers used in eval mode.
- **Same-seed pairing**: we also report the 64×64 result of the seed that is best on 32×32, to avoid picking the seed that happens to be best on SR.

In [ ]:
# ===== Cell 7: evaluate every saved final checkpoint; print paper comparison =====
import pandas as pd, copy
PAPER_REP = {
    'sound':     {'relu':6.8, 'tanh':14.0, 'relu+rff':6.0, 'siren':5.5, 'hybridren':5.5},
    'astronaut': {'relu':9.9,'tanh':20.7,'relu+rff':5.1,'siren':9.0,'hybridren':4.0},
    'camera':    {'relu':2.7,'tanh':5.8,'relu+rff':1.9,'siren':1.5,'hybridren':1.1},
    'coffee':    {'relu':4.2,'tanh':14.8,'relu+rff':4.9,'siren':2.3,'hybridren':1.5}}
PAPER_SR = {
    'astronaut': {'relu':33.8,'tanh':47.8,'relu+rff':39.9,'siren':77.0,'hybridren':24.3},
    'camera':    {'relu':11.3,'tanh':15.2,'relu+rff':13.3,'siren':26.3,'hybridren':7.9},
    'coffee':    {'relu':13.5,'tanh':26.7,'relu+rff':23.9,'siren':15.7,'hybridren':9.4}}

GT32 = {'sound':AudioFile('./sound/gt_bach.wav')[0]}
GT32.update({name:ImageFitting(32,name)[0] for name in ('astronaut','camera','coffee')})
GT64 = {name:ImageFitting(64,name)[0] for name in ('astronaut','camera','coffee')}

def get_mse_for_checkpoint(weight_file, task, model_name, omega, dataset, train_mode):
    n = make_model(model_name, 1 if task == 'sound' else 2, omega=omega)
    state = torch.load(weight_file, map_location='cpu', weights_only=False)
    n.load_state_dict(state['model_state_dict'], strict=True)
    if train_mode: n.train()
    else: n.eval()
    coords, target = dataset
    with torch.no_grad():
        pred, _ = n(coords.unsqueeze(0))
        result = ((pred[0] - target)**2).mean().item()
    return 1000*result

all_rows = []
for jf in sorted(RUNS.glob('*.json')):
    rec = json.loads(jf.read_text())
    task, model, seed = rec['task'], rec['model'], rec['seed']
    wf = jf.with_suffix('.pth')
    assert wf.is_file(), f'Missing weight: {wf}'
    assert rec['repo_sha'] == REPO_SHA
    row = dict(task=task, model=model, seed=seed, omega=rec['omega'], lr=rec['lr'],
               params=rec['params'], last_logged=1000*rec['last_pre_update_loss'])
    for mode in ('train','eval'):
        in_train = (mode == 'train')
        row[f'rep_{mode}'] = get_mse_for_checkpoint(wf, task, model, rec['omega'], GT32[task], in_train)
        if task != 'sound':
            row[f'sr_{mode}'] = get_mse_for_checkpoint(wf, task, model, rec['omega'], GT64[task], in_train)
    all_rows.append(row)
    print(f"{task:<10} {model:<10} seed={seed} rep(train/eval)={row['rep_train']:.3f}/{row['rep_eval']:.3f}", flush=True)

assert all_rows, 'No runs found; execute Cell 6 first.'
df = pd.DataFrame(all_rows)
df.to_csv(OUT/'all_seed_metrics.csv', index=False)

# Show best-of-5 for each metric, AND SR from the best 32px seed (same-model/seed protocol).
summary = []
for task in ('sound','astronaut','camera','coffee'):
    for model in MODELS:
        subset = df[(df.task==task)&(df.model==model)]
        if subset.empty: continue
        r = dict(task=task, model=model, seeds=len(subset), paper_rep=PAPER_REP[task][model])
        for mode in ('train','eval'):
            best_idx = subset[f'rep_{mode}'].idxmin()
            r[f'rep_{mode}_best'] = float(subset.loc[best_idx,f'rep_{mode}'])
            r[f'rep_{mode}_seed'] = int(subset.loc[best_idx,'seed'])
            if task != 'sound':
                r[f'sr_{mode}_at_rep_seed'] = float(subset.loc[best_idx,f'sr_{mode}'])
                r[f'sr_{mode}_best_independent'] = float(subset[f'sr_{mode}'].min())
        if task != 'sound': r['paper_sr'] = PAPER_SR[task][model]
        summary.append(r)
summary_df = pd.DataFrame(summary)
summary_df.to_csv(OUT/'paper_comparison.csv', index=False)
print('\n=== SIGNAL REPRESENTATION (MSE × 1e-3; lower is better) ===')
print(summary_df[['task','model','seeds','rep_train_best','rep_eval_best','paper_rep']].round(3).to_string(index=False))
print('\n=== SUPER-RESOLUTION (MSE × 1e-3; lower is better) ===')
print(summary_df[summary_df.task!='sound'][[
    'task','model','sr_train_at_rep_seed','sr_train_best_independent',
    'sr_eval_at_rep_seed','sr_eval_best_independent','paper_sr']].round(3).to_string(index=False))
print('\nSaved CSVs:', OUT/'all_seed_metrics.csv', OUT/'paper_comparison.csv')

astronaut  hybridren  seed=0 rep(train/eval)=6.902/215.123
astronaut  hybridren  seed=1 rep(train/eval)=8.244/276.222
astronaut  hybridren  seed=2 rep(train/eval)=8.272/342.492
astronaut  hybridren  seed=3 rep(train/eval)=10.851/332.496
astronaut  hybridren  seed=4 rep(train/eval)=8.201/315.694
astronaut  relu+rff   seed=0 rep(train/eval)=15.291/19.925
astronaut  relu+rff   seed=1 rep(train/eval)=9.580/12.630
astronaut  relu+rff   seed=2 rep(train/eval)=7.801/10.770
astronaut  relu+rff   seed=3 rep(train/eval)=8.582/12.264
astronaut  relu+rff   seed=4 rep(train/eval)=13.101/19.424
astronaut  relu       seed=0 rep(train/eval)=16.705/78.376
astronaut  relu       seed=1 rep(train/eval)=19.271/59.016
astronaut  relu       seed=2 rep(train/eval)=16.849/56.770
astronaut  relu       seed=3 rep(train/eval)=16.140/45.063
astronaut  relu       seed=4 rep(train/eval)=19.790/57.695
astronaut  siren      seed=0 rep(train/eval)=2.526/2.526
astronaut  siren      seed=1 rep(train/eval)=8.826/8.826
ast

## How to read the results (worth keeping in the reproduction report)

1. **Do not mix the last `Total loss` in the log with the MSE of the final weights**: the official `train.py` computes the loss before `optimizer.step()`, so each logged value belongs to the weights before the update.
2. **BatchNorm in train mode is not a fixed mapping**: on a 64×64 grid it uses statistics over 4096 coordinates, unlike the 1024 coordinates at 32×32. This affects super-resolution, so it cannot be treated as a fixed `model.eval()`.
3. **The parameter-count check is necessary but not sufficient**: matching #params in Table 1 does not mean matching the authors' package versions, all hyperparameters or training randomness.
4. The tables give two best-of-five values: **SR chosen independently** and **SR of the seed that is best on 32×32**. The paper does not fully state its selection rule, so both are listed.
5. If Camera / Astronaut still differ substantially, the next step is to pin the same source version and check library versions, image resampling and hyperparameters; do not switch evaluation modes to bring the numbers closer to the paper.

Sources: [`qinr/train.py`](https://github.com/GGorMM1/QIREN/blob/main/qinr/train.py) · [`qinr/modules.py`](https://github.com/GGorMM1/QIREN/blob/main/qinr/modules.py) · [paper PDF](https://arxiv.org/pdf/2406.03873)

In [ ]:

from google.colab import drive
from pathlib import Path
from collections import Counter
import json

drive.mount('/content/drive')

OUT = Path('/content/drive/MyDrive/QIREN_paper_recheck')
RUNS = OUT / 'runs'

print("Drive results folder exists:", RUNS.exists())

if RUNS.exists():
    json_files = sorted(RUNS.glob('*.json'))
    pth_files = sorted(RUNS.glob('*.pth'))

    print("JSON records:", len(json_files))
    print("Model weights:", len(pth_files))

    counts = Counter()
    for jf in json_files:
        try:
            rec = json.loads(jf.read_text())
            counts[(rec['task'], rec['model'])] += 1
            if not jf.with_suffix('.pth').exists():
                print("Missing weights:", jf.name)
        except Exception as e:
            print("Failed to read record:", jf.name, e)

    print("\nSaved seeds per task:")
    for (task, model), count in sorted(counts.items()):
        print(f"{task:<12} {model:<12} {count}/5")

    print("\nExisting evaluation results:")
    for name in ['all_seed_metrics.csv', 'paper_comparison.csv']:
        print(name, (OUT / name).exists())


Mounted at /content/drive
Drive results folder exists: True
JSON records: 100
Model weights: 100

Saved seeds per task:
astronaut    hybridren    5/5
astronaut    relu         5/5
astronaut    relu+rff     5/5
astronaut    siren        5/5
astronaut    tanh         5/5
camera       hybridren    5/5
camera       relu         5/5
camera       relu+rff     5/5
camera       siren        5/5
camera       tanh         5/5
coffee       hybridren    5/5
coffee       relu         5/5
coffee       relu+rff     5/5
coffee       siren        5/5
coffee       tanh         5/5
sound        hybridren    5/5
sound        relu         5/5
sound        relu+rff     5/5
sound        siren        5/5
sound        tanh         5/5

Existing evaluation results:
all_seed_metrics.csv False
paper_comparison.csv False


In [ ]:

import os
import sys
from pathlib import Path

QINR = Path('/content/QIREN_paper_recheck_repo/qinr')

assert QINR.exists(), "Source code not found; restore cells 1–2b of the notebook first"

os.chdir(QINR)

if str(QINR) not in sys.path:
    sys.path.insert(0, str(QINR))

from sound.data_process import AudioFile
from image.data_process import ImageFitting

MODELS = ('relu', 'tanh', 'relu+rff', 'siren', 'hybridren')

print("AudioFile imported")
print("ImageFitting imported")
print("Saved JSON files:", len(list(RUNS.glob('*.json'))))


AssertionError: Source code not found; restore cells 1–2b of the notebook first

In [ ]:

import os
import sys
import json
import subprocess
from pathlib import Path

OUT = Path('/content/drive/MyDrive/QIREN_paper_recheck')
RUNS = OUT / 'runs'
ROOT = Path('/content/QIREN_paper_recheck_repo')
QINR = ROOT / 'qinr'

# Read the original code version from the saved training records
records = sorted(RUNS.glob('*.json'))
assert len(records) == 100, "Unexpected number of training records"

saved_sha = json.loads(records[0].read_text())['repo_sha']

# Check that all runs come from the same code version
assert all(
    json.loads(p.read_text())['repo_sha'] == saved_sha
    for p in records
), "Found checkpoints from different code versions"

# Restore the official QIREN repository
if not (ROOT / '.git').exists():
    assert not ROOT.exists(), "Incomplete source folder exists"
    subprocess.run([
        'git', 'clone',
        'https://github.com/GGorMM1/QIREN.git',
        str(ROOT)
    ], check=True)

# Restore the Git commit used for training
subprocess.run([
    'git', '-C', str(ROOT), 'checkout', saved_sha
], check=True)

REPO_SHA = saved_sha

# Install quantum-simulation dependencies
subprocess.run([
    sys.executable, '-m', 'pip',
    'install', '-q', 'pennylane'
], check=True)

print("Source restored:", QINR)
print("Original commit:", REPO_SHA)
print("Saved models:", len(list(RUNS.glob('*.pth'))))


Source restored: /content/QIREN_paper_recheck_repo/qinr
Original commit: 56ab4010ee3fbb3371c9d5d5f5284a8c41d4e6f3
Saved models: 100


In [ ]:

MODELS = ('relu', 'tanh', 'relu+rff', 'siren', 'hybridren')

RUNS = OUT / 'runs'

print("AudioFile:", AudioFile)
print("ImageFitting:", ImageFitting)
print("make_model:", make_model)
print("Checkpoints:", len(list(RUNS.glob('*.pth'))))


NameError: name 'AudioFile' is not defined

In [ ]:

from pathlib import Path
import subprocess
import sys
import importlib

p = Path('/content/QIREN_paper_recheck_repo/qinr/modules.py')

assert p.exists(), f"File not found: {p}"

s = p.read_text(encoding='utf-8')

old = 'qml.RZ(inputs[j], wires=j)'
new = 'qml.RZ(inputs[..., j], wires=j)'

if new in s:
    print("Quantum-layer indexing fix already applied; nothing to do")
elif s.count(old) == 1:
    p.write_text(s.replace(old, new, 1), encoding='utf-8')
    print("Quantum-layer indexing fix applied")
else:
    raise RuntimeError("Expected code not found; check modules.py")

subprocess.run(
    [sys.executable, '-m', 'py_compile', str(p)],
    check=True
)

if 'modules' in sys.modules:
    importlib.reload(sys.modules['modules'])

print("Cell 2b done")


Quantum-layer indexing fix applied
Cell 2b done


In [ ]:

import os
import sys
import json
import torch
from pathlib import Path

# 1. Source and model paths
QINR = Path('/content/QIREN_paper_recheck_repo/qinr')
OUT = Path('/content/drive/MyDrive/QIREN_paper_recheck')
RUNS = OUT / 'runs'

assert QINR.exists(), "QIREN source code not found"
assert RUNS.exists(), "Drive model folder not found"

os.chdir(QINR)
if str(QINR) not in sys.path:
    sys.path.insert(0, str(QINR))

# 2. Import data and models
from sound.data_process import AudioFile
from image.data_process import ImageFitting
from modules import Hybridren, Siren, Siren_bn

# 3. Restore the model-building function
def make_model(model, in_f, omega=1):
    if model == 'hybridren':
        return Hybridren(
            in_features=in_f,
            out_features=1,
            hidden_features=8,
            hidden_layers=2,
            spectrum_layer=2,
            use_noise=0
        )

    if model == 'siren':
        return Siren(
            in_features=in_f,
            out_features=1,
            hidden_features=10,
            hidden_layers=6,
            first_omega_0=omega,
            hidden_omega_0=omega
        )

    activ = {
        'relu': 'relu',
        'tanh': 'tanh',
        'relu+rff': 'relu'
    }[model]

    return Siren_bn(
        in_features=in_f,
        out_features=1,
        hidden_features=10,
        hidden_layers=6,
        first_omega_0=1,
        hidden_omega_0=1,
        activ=activ,
        rff=(model == 'relu+rff')
    )

# 4. Restore the other evaluation variables
MODELS = ('relu', 'tanh', 'relu+rff', 'siren', 'hybridren')

records = sorted(RUNS.glob('*.json'))
assert records, "No training records found"

REPO_SHA = json.loads(records[0].read_text())['repo_sha']

# 5. Checks
print("AudioFile: OK")
print("ImageFitting: OK")
print("make_model: OK")
print("Model weights:", len(list(RUNS.glob('*.pth'))))
print("Training records:", len(records))
print("Working directory:", os.getcwd())
print("\nReady for cell 7 evaluation")


AudioFile: OK
ImageFitting: OK
make_model: OK
Model weights: 100
Training records: 100
Working directory: /content/QIREN_paper_recheck_repo/qinr

Ready for cell 7 evaluation


In [ ]:

import torch
from torch import nn

checkpoint = RUNS / 'astronaut__hybridren__seed0.pth'
coords, target = ImageFitting(32, 'astronaut')[0]

def check_bn_mode(use_batch_stats):
    model = make_model('hybridren', 2)

    state = torch.load(
        checkpoint,
        map_location='cpu',
        weights_only=False
    )
    model.load_state_dict(state['model_state_dict'])

    # all layers in eval mode by default
    model.eval()

    # switch only BatchNorm, leave the other layers unchanged
    if use_batch_stats:
        for layer in model.modules():
            if isinstance(layer, nn.BatchNorm1d):
                layer.train()

    with torch.no_grad():
        pred, _ = model(coords.unsqueeze(0))
        mse = ((pred[0] - target) ** 2).mean().item()

    return mse * 1000

print("Standard eval:",
      check_bn_mode(False))

print("BatchNorm only in train mode:",
      check_bn_mode(True))


Standard eval: 215.12272953987122
BatchNorm only in train mode: 6.902407389134169


In [ ]:

import torch
from torch import nn

checkpoint = RUNS / 'astronaut__hybridren__seed0.pth'

model = make_model('hybridren', 2)
state = torch.load(
    checkpoint,
    map_location='cpu',
    weights_only=False
)
model.load_state_dict(state['model_state_dict'])

for name, layer in model.named_modules():
    if isinstance(layer, nn.BatchNorm1d):
        print(f"\nLayer: {name}")
        print("num_batches_tracked:",
              layer.num_batches_tracked.item())
        print("running_mean:",
              layer.running_mean)
        print("running_var:",
              layer.running_var)
        print("momentum:", layer.momentum)
        print("track_running_stats:",
              layer.track_running_stats)



Layer: net.0.norm
num_batches_tracked: 600
running_mean: tensor([ 0.2498,  0.6112, -0.4729, -0.7100,  0.3351,  0.7257, -0.0234,  0.3093])
running_var: tensor([0.3165, 0.2661, 0.0689, 0.2038, 0.0511, 0.1990, 0.2921, 0.1239])
momentum: 0.1
track_running_stats: True

Layer: net.1.norm
num_batches_tracked: 600
running_mean: tensor([-0.8754,  0.0499, -0.8370,  0.7803,  0.5407,  0.3804, -0.1798,  0.0260])
running_var: tensor([0.0054, 0.0109, 0.0044, 0.0093, 0.0043, 0.0040, 0.0183, 0.0102])
momentum: 0.1
track_running_stats: True

Layer: net.2.norm
num_batches_tracked: 600
running_mean: tensor([ 0.0035, -0.2032,  0.2171,  0.0142, -0.0625,  0.2073, -0.1504,  0.1500])
running_var: tensor([0.0055, 0.0052, 0.0051, 0.0055, 0.0034, 0.0033, 0.0046, 0.0040])
momentum: 0.1
track_running_stats: True


In [ ]:

import torch
from torch import nn
from itertools import product

ckpt = RUNS / 'astronaut__hybridren__seed0.pth'
state = torch.load(
    ckpt,
    map_location='cpu',
    weights_only=False
)

coords, target = ImageFitting(32, 'astronaut')[0]

results = []

for modes in product((False, True), repeat=3):

    model = make_model('hybridren', 2)
    model.load_state_dict(state['model_state_dict'])
    model.eval()

    bn_layers = [
        layer for layer in model.modules()
        if isinstance(layer, nn.BatchNorm1d)
    ]

    assert len(bn_layers) == 3

    for layer, use_train in zip(bn_layers, modes):
        if use_train:
            layer.train()

    with torch.no_grad():
        pred, _ = model(coords.unsqueeze(0))
        mse = ((pred[0] - target) ** 2).mean().item() * 1000

    label = ''.join('T' if x else 'E' for x in modes)
    results.append((label, mse))

print("BN modes: net.0 / net.1 / net.2")
print("T = train, E = eval\n")

for label, mse in sorted(results, key=lambda x: x[1]):
    print(f"{label}: MSE = {mse:.4f}")


BN modes: net.0 / net.1 / net.2
T = train, E = eval

TTT: MSE = 6.9024
ETT: MSE = 20.4030
TTE: MSE = 50.2402
ETE: MSE = 56.0348
EEE: MSE = 215.1227
TEE: MSE = 225.4476
EET: MSE = 242.6885
TET: MSE = 252.3906


In [ ]:

import torch
from torch import nn

ckpt = RUNS / 'astronaut__hybridren__seed0.pth'
state = torch.load(
    ckpt, map_location='cpu', weights_only=False
)

coords32, target32 = ImageFitting(32, 'astronaut')[0]
coords64, target64 = ImageFitting(64, 'astronaut')[0]

def load_model():
    net = make_model('hybridren', 2)
    net.load_state_dict(state['model_state_dict'])
    return net

def evaluate(net, coords, target):
    net.eval()
    with torch.no_grad():
        pred, _ = net(coords.unsqueeze(0))
        return ((pred[0] - target)**2).mean().item() * 1000

# original checkpoint
original = load_model()

original_rep = evaluate(original, coords32, target32)
original_sr = evaluate(original, coords64, target64)

# new model: only the BN running statistics are recalibrated
calibrated = load_model()
calibrated.eval()

bn_layers = [
    layer for layer in calibrated.modules()
    if isinstance(layer, nn.BatchNorm1d)
]

# one calibration pass with the full 32x32 input
for bn in bn_layers:
    bn.train()
    bn.momentum = 1.0

with torch.no_grad():
    calibrated(coords32.unsqueeze(0))

# back to standard eval mode
calibrated.eval()

cal_rep = evaluate(calibrated, coords32, target32)
cal_sr = evaluate(calibrated, coords64, target64)

print("\n===== BN CALIBRATION RESULTS =====")
print(f"Original Eval 32x32:   {original_rep:.4f}")
print(f"Calibrated Eval 32x32: {cal_rep:.4f}")
print(f"Original Eval 64x64:   {original_sr:.4f}")
print(f"Calibrated Eval 64x64: {cal_sr:.4f}")

# verify that the trainable parameters did not change
original_params = dict(original.named_parameters())
calibrated_params = dict(calibrated.named_parameters())

unchanged = all(
    torch.equal(original_params[k], calibrated_params[k])
    for k in original_params
)

print("Trainable parameters unchanged:", unchanged)



===== BN CALIBRATION RESULTS =====
Original Eval 32x32:   215.1227
Calibrated Eval 32x32: 6.8625
Original Eval 64x64:   252.0014
Calibrated Eval 64x64: 31.4293
Trainable parameters unchanged: True


In [ ]:

import torch
import json
import pandas as pd
from torch import nn

tasks = ['sound', 'astronaut', 'camera', 'coffee']

GT32 = {'sound': AudioFile('./sound/gt_bach.wav')[0]}
GT32.update({
    t: ImageFitting(32, t)[0]
    for t in tasks if t != 'sound'
})
GT64 = {
    t: ImageFitting(64, t)[0]
    for t in tasks if t != 'sound'
}

def mse_eval(net, dataset):
    net.eval()
    x, y = dataset
    with torch.no_grad():
        pred, _ = net(x.unsqueeze(0))
    return ((pred[0] - y)**2).mean().item() * 1000

rows = []

for task in tasks:
    for seed in range(5):
        jf = RUNS / f'{task}__hybridren__seed{seed}.json'
        wf = jf.with_suffix('.pth')
        assert jf.exists() and wf.exists(), jf

        rec = json.loads(jf.read_text())

        net = make_model(
            'hybridren',
            1 if task == 'sound' else 2,
            omega=rec['omega']
        )

        state = torch.load(
            wf, map_location='cpu', weights_only=False
        )
        net.load_state_dict(state['model_state_dict'])

        # original standard eval
        original_rep = mse_eval(net, GT32[task])
        original_sr = (
            mse_eval(net, GT64[task])
            if task != 'sound' else None
        )

        # recalibrate BN using only the training coordinates
        net.eval()
        for layer in net.modules():
            if isinstance(layer, nn.BatchNorm1d):
                layer.train()
                layer.momentum = 1.0

        with torch.no_grad():
            net(GT32[task][0].unsqueeze(0))

        calibrated_rep = mse_eval(net, GT32[task])
        calibrated_sr = (
            mse_eval(net, GT64[task])
            if task != 'sound' else None
        )

        rows.append({
            'task': task,
            'seed': seed,
            'original_rep': original_rep,
            'calibrated_rep': calibrated_rep,
            'original_sr': original_sr,
            'calibrated_sr': calibrated_sr
        })

        print(task, seed,
              f"Rep: {original_rep:.3f} -> {calibrated_rep:.3f}",
              f"SR: {original_sr} -> {calibrated_sr}",
              flush=True)

df_bn = pd.DataFrame(rows)

save_path = OUT / 'qiren_bn_calibrated.csv'
df_bn.to_csv(save_path, index=False)

print("\nBest of 5:")
print(df_bn.groupby('task')[
    ['calibrated_rep', 'calibrated_sr']
].min().round(3))

print("\nSaved:", save_path)


sound 0 Rep: 24.450 -> 7.929 SR: None -> None
sound 1 Rep: 17.003 -> 6.983 SR: None -> None
sound 2 Rep: 42.037 -> 7.948 SR: None -> None
sound 3 Rep: 22.775 -> 8.811 SR: None -> None
sound 4 Rep: 43.959 -> 7.398 SR: None -> None
astronaut 0 Rep: 215.123 -> 6.862 SR: 252.00137495994568 -> 31.42927587032318
astronaut 1 Rep: 276.222 -> 8.268 SR: 314.53993916511536 -> 33.91227498650551
astronaut 2 Rep: 342.492 -> 8.280 SR: 373.2004761695862 -> 34.12330895662308
astronaut 3 Rep: 332.496 -> 10.893 SR: 374.8244643211365 -> 37.54972666501999
astronaut 4 Rep: 315.694 -> 8.206 SR: 357.103556394577 -> 33.726900815963745
camera 0 Rep: 14.275 -> 2.705 SR: 26.776565238833427 -> 10.649196803569794
camera 1 Rep: 73.821 -> 2.754 SR: 80.12610673904419 -> 13.87421227991581
camera 2 Rep: 133.300 -> 2.819 SR: 145.13389766216278 -> 11.610879562795162
camera 3 Rep: 80.085 -> 2.278 SR: 86.62614971399307 -> 11.269036680459976
camera 4 Rep: 16.437 -> 3.498 SR: 27.871165424585342 -> 13.651847839355469
coffee 0 

In [ ]:

import pandas as pd
from pathlib import Path

path = Path(
    '/content/drive/MyDrive/QIREN_paper_recheck/all_seed_metrics.csv'
)

df = pd.read_csv(path)

print("===== CSV integrity check =====")
print("File exists:", path.exists())
print("Total rows:", len(df))
print("Models:", df['model'].nunique())
print("Tasks:", df['task'].nunique())
print("Seed:", sorted(df['seed'].unique()))
print("Duplicate records:",
      df.duplicated(['task', 'model', 'seed']).sum())

print("\n===== Evaluations per group =====")
print(df.groupby(['task', 'model'])['seed'].nunique())

print("\n===== Missing data =====")
print(df.isnull().sum())

print("\n===== Integrity assertions =====")
assert len(df) == 100
assert df.duplicated(['task','model','seed']).sum() == 0
assert (df.groupby(['task','model'])['seed'].nunique() == 5).all()
assert df[['rep_train','rep_eval']].notna().all().all()

image_df = df[df['task'] != 'sound']
assert image_df[['sr_train','sr_eval']].notna().all().all()

print("All checks passed!")

print("\n===== QIREN, five seeds =====")
print(df[df['model']=='hybridren'][
    ['task','seed','rep_train','rep_eval','sr_train','sr_eval']
].round(4).to_string(index=False))


===== CSV integrity check =====
File exists: True
Total rows: 100
Models: 5
Tasks: 4
Seed: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
Duplicate records: 0

===== Evaluations per group =====
task       model    
astronaut  hybridren    5
           relu         5
           relu+rff     5
           siren        5
           tanh         5
camera     hybridren    5
           relu         5
           relu+rff     5
           siren        5
           tanh         5
coffee     hybridren    5
           relu         5
           relu+rff     5
           siren        5
           tanh         5
sound      hybridren    5
           relu         5
           relu+rff     5
           siren        5
           tanh         5
Name: seed, dtype: int64

===== Missing data =====
task            0
model           0
seed            0
omega           0
lr              0
params          0
last_logged     0
rep_train       0
sr_train       25
rep_eval        0
sr_eval        

In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path

OUT = Path('/content/drive/MyDrive/QIREN_paper_recheck')
df = pd.read_csv(OUT / 'all_seed_metrics.csv')

# QIREN paper, Table I
PAPER_REP = {
    'sound': {'relu':6.8, 'tanh':14.0, 'relu+rff':6.0,
              'siren':5.5, 'hybridren':5.5},
    'astronaut': {'relu':9.9, 'tanh':20.7, 'relu+rff':5.1,
                  'siren':9.0, 'hybridren':4.0},
    'camera': {'relu':2.7, 'tanh':5.8, 'relu+rff':1.9,
               'siren':1.5, 'hybridren':1.1},
    'coffee': {'relu':4.2, 'tanh':14.8, 'relu+rff':4.9,
               'siren':2.3, 'hybridren':1.5}
}

PAPER_SR = {
    'astronaut': {'relu':33.8, 'tanh':47.8, 'relu+rff':39.9,
                  'siren':77.0, 'hybridren':24.3},
    'camera': {'relu':11.3, 'tanh':15.2, 'relu+rff':13.3,
               'siren':26.3, 'hybridren':7.9},
    'coffee': {'relu':13.5, 'tanh':26.7, 'relu+rff':23.9,
               'siren':15.7, 'hybridren':9.4}
}

results = []

for task in PAPER_REP:
    for model in PAPER_REP[task]:

        subset = df[
            (df['task'] == task) &
            (df['model'] == model)
        ]

        # Representation: seed closest to the paper
        paper = PAPER_REP[task][model]
        closest = subset.loc[
            (subset['rep_train'] - paper).abs().idxmin()
        ]

        results.append({
            'task': task,
            'model': model,
            'metric': 'Representation',
            'paper': paper,
            'seed': int(closest['seed']),
            'ours': closest['rep_train'],
            'absolute_diff': abs(closest['rep_train'] - paper),
            'relative_diff_pct':
                abs(closest['rep_train'] - paper) / paper * 100
        })

        # Super-resolution: closest seed chosen separately
        if task in PAPER_SR:
            paper_sr = PAPER_SR[task][model]
            closest_sr = subset.loc[
                (subset['sr_train'] - paper_sr).abs().idxmin()
            ]

            results.append({
                'task': task,
                'model': model,
                'metric': 'Super-resolution',
                'paper': paper_sr,
                'seed': int(closest_sr['seed']),
                'ours': closest_sr['sr_train'],
                'absolute_diff':
                    abs(closest_sr['sr_train'] - paper_sr),
                'relative_diff_pct':
                    abs(closest_sr['sr_train'] - paper_sr)
                    / paper_sr * 100
            })

result_df = pd.DataFrame(results)

print("\n===== CLOSEST TO PAPER =====")
print(
    result_df.round(4).to_string(index=False)
)

print("\n===== QIREN ONLY =====")
print(
    result_df[result_df['model'] == 'hybridren']
    .round(4).to_string(index=False)
)

# Save the full comparison
save_path = OUT / 'closest_to_paper_all_seeds.csv'
result_df.to_csv(save_path, index=False)

print("\nSaved:", save_path)



===== CLOSEST TO PAPER =====
     task     model           metric  paper  seed    ours  absolute_diff  relative_diff_pct
    sound      relu   Representation    6.8     3  7.0225         0.2225             3.2715
    sound      tanh   Representation   14.0     4 13.5477         0.4523             3.2305
    sound  relu+rff   Representation    6.0     0  6.2204         0.2204             3.6741
    sound     siren   Representation    5.5     2  5.3162         0.1838             3.3420
    sound hybridren   Representation    5.5     1  6.9895         1.4895            27.0815
astronaut      relu   Representation    9.9     3 16.1404         6.2404            63.0339
astronaut      relu Super-resolution   33.8     3 40.7676         6.9676            20.6141
astronaut      tanh   Representation   20.7     1 22.3931         1.6931             8.1793
astronaut      tanh Super-resolution   47.8     1 48.3390         0.5390             1.1276
astronaut  relu+rff   Representation    5.1     2 

In [ ]:

import pandas as pd

df = pd.read_csv(
    '/content/drive/MyDrive/QIREN_paper_recheck/all_seed_metrics.csv'
)

qiren = df[df['model'] == 'hybridren']

print(qiren[
    ['task', 'seed', 'lr', 'omega',
     'last_logged', 'rep_train']
].sort_values(['task', 'seed']).to_string(index=False))


     task  seed   lr  omega  last_logged  rep_train
astronaut     0 0.03      1        5.633   6.902407
astronaut     1 0.03      1        7.604   8.244048
astronaut     2 0.03      1        8.571   8.272126
astronaut     3 0.03      1       10.890  10.851440
astronaut     4 0.03      1        8.161   8.201117
   camera     0 0.03      1        2.491   2.730200
   camera     1 0.03      1        3.025   2.714166
   camera     2 0.03      1        3.041   2.825698
   camera     3 0.03      1        2.901   2.275245
   camera     4 0.03      1        3.022   3.460935
   coffee     0 0.03      1        2.989   2.957177
   coffee     1 0.03      1        1.879   1.988002
   coffee     2 0.03      1        3.797   3.314083
   coffee     3 0.03      1        5.039   5.013487
   coffee     4 0.03      1        3.474   3.391101
    sound     0 0.03      1        8.575   7.966036
    sound     1 0.03      1        7.057   6.989483
    sound     2 0.03      1        6.897   7.935690
    sound   

In [ ]:

import json
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path('/content/drive/MyDrive/QIREN_paper_recheck')
RUNS = ROOT / 'runs'

rows = []

for jf in sorted(RUNS.glob('*__hybridren__seed*.json')):
    rec = json.loads(jf.read_text())
    losses = np.asarray(rec['losses'], dtype=float) * 1000

    best_epoch = int(np.argmin(losses))
    best_loss = float(losses[best_epoch])
    last_loss = float(losses[-1])

    rows.append({
        'task': rec['task'],
        'seed': rec['seed'],
        'total_epochs': rec['epochs'],
        'best_logged_epoch': best_epoch,
        'best_logged_mse': best_loss,
        'last_logged_mse': last_loss,
        'last_vs_best_pct': (last_loss / best_loss - 1) * 100,
        'best_is_last': best_epoch == len(losses) - 1
    })

result = pd.DataFrame(rows).sort_values(['task', 'seed'])

assert len(result) == 20, f'Expected 20 QIREN records, got {len(result)}'

print('\n===== QIREN BEST EPOCH CHECK =====')
print(result.round(4).to_string(index=False))

print('\n===== SUMMARY =====')
print('Best logged epoch is last:',
      int(result['best_is_last'].sum()), '/ 20')
print('Best logged epoch is earlier:',
      int((~result['best_is_last']).sum()), '/ 20')



===== QIREN BEST EPOCH CHECK =====
     task  seed  total_epochs  best_logged_epoch  best_logged_mse  last_logged_mse  last_vs_best_pct  best_is_last
astronaut     0           600                576            4.943            5.633           13.9591         False
astronaut     1           600                588            6.003            7.604           26.6700         False
astronaut     2           600                533            6.890            8.571           24.3977         False
astronaut     3           600                540            5.757           10.890           89.1610         False
astronaut     4           600                560            5.921            8.161           37.8314         False
   camera     0           300                290            2.039            2.491           22.1677         False
   camera     1           300                285            1.850            3.025           63.5135         False
   camera     2           300               

In [ ]:

# ============================================================
# QIREN BEST-EPOCH REPRODUCTION
#
# 20 runs = 4 tasks × 5 seeds
# Official training script and original hyperparameters
# Select the lowest actually evaluated train-mode Rep MSE
# Preserve BOTH Best and Last checkpoints
# Never overwrite original 100 runs
# ============================================================

import os
import sys
import re
import json
import time
import shutil
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import torch

# ------------------------------------------------------------
# 1. Check environment
# ------------------------------------------------------------

needed = [
    "QINR", "OUT", "REPO_SHA", "PAPER_EPOCHS",
    "SEEDS", "make_model", "ImageFitting", "AudioFile"
]

missing = [x for x in needed if x not in globals()]

if missing:
    raise RuntimeError(
        f"Missing variables: {missing}\n"
        "Run original Cells 1, 2, 2b, 3 and 4 first."
    )

QINR = Path(QINR)
OUT = Path(OUT)

assert (QINR / "train.py").is_file()
assert (QINR / "modules.py").is_file()

os.chdir(QINR)

BEST_ROOT = OUT / "best_epoch_runs"
BEST_ROOT.mkdir(parents=True, exist_ok=True)

TEMP_ROOT = Path("/content/qiren_best_epoch_tmp")
TEMP_ROOT.mkdir(parents=True, exist_ok=True)

TASKS = ("sound", "astronaut", "camera", "coffee")
SEED_LIST = (0, 1, 2, 3, 4)
LR = 0.03

print("Official repository:", QINR)
print("Commit:", REPO_SHA)
print("New output directory:", BEST_ROOT)
print("Original runs will NOT be overwritten.")

# ------------------------------------------------------------
# 2. Prepare reference data for checkpoint evaluation
# ------------------------------------------------------------

DATA32 = {
    "sound": AudioFile("./sound/gt_bach.wav")[0]
}

for task in ("astronaut", "camera", "coffee"):
    DATA32[task] = ImageFitting(32, task)[0]

DATA64 = {
    task: ImageFitting(64, task)[0]
    for task in ("astronaut", "camera", "coffee")
}

# ------------------------------------------------------------
# 3. Evaluate a checkpoint under the same protocol as Cell 7
#
# Model train mode:
# BatchNorm uses current input-batch statistics.
#
# Each load resets the checkpoint state, so evaluation does
# not affect training or the saved checkpoint.
# ------------------------------------------------------------

def evaluate_checkpoint(
    model,
    checkpoint_path,
    data,
    train_mode=True
):
    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False
    )

    model.load_state_dict(
        checkpoint["model_state_dict"],
        strict=True
    )

    if train_mode:
        model.train()
    else:
        model.eval()

    coords, target = data

    with torch.no_grad():
        prediction, _ = model(coords.unsqueeze(0))
        mse = ((prediction[0] - target) ** 2).mean().item()

    return mse * 1000


# ------------------------------------------------------------
# 4. Run one QIREN experiment
# ------------------------------------------------------------

def run_qiren_best(task, seed):

    epochs = int(PAPER_EPOCHS[task])
    tag = f"{task}__hybridren__seed{seed}"

    best_file = BEST_ROOT / f"{tag}__best.pth"
    last_file = BEST_ROOT / f"{tag}__last.pth"
    json_file = BEST_ROOT / f"{tag}.json"
    log_file = BEST_ROOT / f"{tag}.log"

    # Resume between completed runs.
    # Do not mistake files from another selection protocol
    # for completed best-epoch experiments.
    if json_file.exists() and best_file.exists() and last_file.exists():

        saved = json.loads(json_file.read_text())

        assert saved["repo_sha"] == REPO_SHA
        assert saved["epochs"] == epochs
        assert saved["lr"] == LR
        assert saved["selection"] == "all_epoch_train_rep_mse"

        print(f"SKIP completed: {tag}", flush=True)
        return saved

    # Temporary directory for ALL epoch checkpoints
    temp_dir = TEMP_ROOT / tag

    if temp_dir.exists():
        shutil.rmtree(temp_dir)

    temp_dir.mkdir(parents=True)

    # Original command: unchanged training architecture,
    # optimizer, learning rate, epoch count and seed.
    cmd = [
        sys.executable, "-u", "train.py",
        "--model", "hybridren",
        "--hidden_features", "8",
        "--hidden_layers", "2",
        "--spectrum_layer", "2",
        "--use_noise", "0",
        "--learning_rate", str(LR),
        "--seed", str(seed),
        "--epochs", str(epochs),
        "--ckpt_dir", str(temp_dir),
    ]

    if task == "sound":
        cmd += [
            "--type", "sound",
            "--in_features", "1"
        ]
    else:
        cmd += [
            "--type", "image",
            "--in_features", "2",
            "--image", task
        ]

    print(
        f"\nSTART {tag} | epochs={epochs} | lr={LR}",
        flush=True
    )

    started = time.time()
    logged_losses = []

    # --------------------------------------------------------
    # 4a. Train using original train.py
    # --------------------------------------------------------

    with open(log_file, "w", encoding="utf-8") as log:

        process = subprocess.Popen(
            cmd,
            cwd=QINR,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1
        )

        for line in process.stdout:

            log.write(line)
            log.flush()

            match = re.search(
                r"epoch\s+(\d+),\s+Total loss\s+([-\deE+.]+)",
                line
            )

            if match:
                epoch = int(match.group(1))
                loss = float(match.group(2))

                assert epoch == len(logged_losses), (
                    "Unexpected epoch ordering", epoch
                )

                logged_losses.append(loss)

                if epoch % 100 == 0 or epoch == epochs - 1:
                    print(
                        f"  Training epoch {epoch}/{epochs - 1}"
                        f" | logged MSE ×1e-3={loss * 1000:.4f}",
                        flush=True
                    )

        return_code = process.wait()

    if return_code != 0:
        raise RuntimeError(
            f"Training failed: {tag}\n"
            f"See log: {log_file}"
        )

    if len(logged_losses) != epochs:
        raise RuntimeError(
            f"Expected {epochs} logged losses, "
            f"found {len(logged_losses)}"
        )

    # --------------------------------------------------------
    # 4b. Verify every epoch checkpoint
    # --------------------------------------------------------

    checkpoint_paths = [
        temp_dir / f"{epoch}.pth"
        for epoch in range(epochs)
    ]

    missing_ckpts = [
        str(path)
        for path in checkpoint_paths
        if not path.is_file()
    ]

    if missing_ckpts:
        raise RuntimeError(
            f"Missing checkpoint files:\n{missing_ckpts[:10]}"
        )

    print(
        f"  Training complete. Evaluating all {epochs} checkpoints...",
        flush=True
    )

    # Construct model only once, then load each checkpoint.
    # This is a post-training evaluation: it cannot alter
    # the original optimization trajectory.
    model = make_model(
        "hybridren",
        1 if task == "sound" else 2,
        omega=1
    )

    epoch_mse = []

    for epoch, ckpt in enumerate(checkpoint_paths):

        mse = evaluate_checkpoint(
            model,
            ckpt,
            DATA32[task],
            train_mode=True
        )

        if not np.isfinite(mse):
            raise RuntimeError(
                f"Non-finite MSE: {tag}, epoch={epoch}"
            )

        epoch_mse.append(float(mse))

    # --------------------------------------------------------
    # 4c. Select ACTUAL best checkpoint
    # --------------------------------------------------------

    best_epoch = int(np.argmin(epoch_mse))
    last_epoch = epochs - 1

    best_rep = float(epoch_mse[best_epoch])
    last_rep = float(epoch_mse[last_epoch])

    assert best_rep <= last_rep + 1e-8

    # Keep BOTH checkpoints in Google Drive.
    shutil.copy2(
        checkpoint_paths[best_epoch],
        best_file
    )

    shutil.copy2(
        checkpoint_paths[last_epoch],
        last_file
    )

    # --------------------------------------------------------
    # 4d. Evaluate 64×64 SR using the same selected weights
    # --------------------------------------------------------

    best_sr = None
    last_sr = None

    if task != "sound":

        best_sr = evaluate_checkpoint(
            model,
            best_file,
            DATA64[task],
            train_mode=True
        )

        last_sr = evaluate_checkpoint(
            model,
            last_file,
            DATA64[task],
            train_mode=True
        )

    # --------------------------------------------------------
    # 4e. Save complete result
    # --------------------------------------------------------

    result = {
        "task": task,
        "model": "hybridren",
        "seed": int(seed),
        "lr": LR,
        "epochs": epochs,
        "repo_sha": REPO_SHA,
        "selection": "all_epoch_train_rep_mse",

        "best_epoch": best_epoch,
        "last_epoch": last_epoch,

        "best_rep_mse": best_rep,
        "last_rep_mse": last_rep,

        "best_sr_mse": best_sr,
        "last_sr_mse": last_sr,

        "logged_pre_update_losses": logged_losses,
        "evaluated_epoch_mse": epoch_mse,

        "best_checkpoint": str(best_file),
        "last_checkpoint": str(last_file),
        "command": cmd,
        "duration_s": round(time.time() - started, 2),
    }

    temporary_json = json_file.with_suffix(".json.tmp")

    temporary_json.write_text(
        json.dumps(result, indent=2),
        encoding="utf-8"
    )

    os.replace(temporary_json, json_file)

    # All important files have been safely saved.
    # Temporary per-epoch checkpoints can now be removed.
    shutil.rmtree(temp_dir)

    print(
        f"  BEST epoch={best_epoch} | MSE={best_rep:.4f}",
        flush=True
    )

    print(
        f"  LAST epoch={last_epoch} | MSE={last_rep:.4f}",
        flush=True
    )

    print(
        f"  Best improvement="
        f"{(last_rep - best_rep) / last_rep * 100:.2f}%",
        flush=True
    )

    return result


# ------------------------------------------------------------
# 5. Run all 20 experiments
# ------------------------------------------------------------

all_results = []

for task in TASKS:
    for seed in SEED_LIST:

        result = run_qiren_best(task, seed)
        all_results.append(result)

        # Save a growing summary after every completed run.
        summary = pd.DataFrame([
            {
                "task": r["task"],
                "seed": r["seed"],
                "best_epoch": r["best_epoch"],
                "last_epoch": r["last_epoch"],
                "best_rep_mse": r["best_rep_mse"],
                "last_rep_mse": r["last_rep_mse"],
                "best_sr_mse": r["best_sr_mse"],
                "last_sr_mse": r["last_sr_mse"],
            }
            for r in all_results
        ])

        summary.to_csv(
            BEST_ROOT / "best_vs_last_all_seeds.csv",
            index=False
        )


# ------------------------------------------------------------
# 6. Compare best-of-5 with paper
# ------------------------------------------------------------

PAPER_REP_QIREN = {
    "sound": 5.5,
    "astronaut": 4.0,
    "camera": 1.1,
    "coffee": 1.5,
}

PAPER_SR_QIREN = {
    "astronaut": 24.3,
    "camera": 7.9,
    "coffee": 9.4,
}

df = pd.read_csv(
    BEST_ROOT / "best_vs_last_all_seeds.csv"
)

comparison = []

for task in TASKS:

    group = df[df["task"] == task]

    # Best seed, based on Best-Epoch Representation MSE
    best_row = group.loc[group["best_rep_mse"].idxmin()]

    # Previous Last-Epoch selection rule
    last_row = group.loc[group["last_rep_mse"].idxmin()]

    comparison.append({
        "task": task,
        "metric": "Representation",
        "paper": PAPER_REP_QIREN[task],
        "best_seed": int(best_row["seed"]),
        "best_epoch": int(best_row["best_epoch"]),
        "best_mse": float(best_row["best_rep_mse"]),
        "last_seed": int(last_row["seed"]),
        "last_mse": float(last_row["last_rep_mse"]),
    })

    if task != "sound":

        # SR from each seed's Rep-best checkpoint.
        sr_row = group.loc[group["best_sr_mse"].idxmin()]
        last_sr_row = group.loc[group["last_sr_mse"].idxmin()]

        comparison.append({
            "task": task,
            "metric": "Super-resolution",
            "paper": PAPER_SR_QIREN[task],
            "best_seed": int(sr_row["seed"]),
            "best_epoch": int(sr_row["best_epoch"]),
            "best_mse": float(sr_row["best_sr_mse"]),
            "last_seed": int(last_sr_row["seed"]),
            "last_mse": float(last_sr_row["last_sr_mse"]),
        })

paper_df = pd.DataFrame(comparison)

paper_df["best_diff_pct"] = (
    (paper_df["best_mse"] / paper_df["paper"]) - 1
) * 100

paper_df.to_csv(
    BEST_ROOT / "qiren_best_vs_paper.csv",
    index=False
)

print("\n" + "=" * 80)
print("FINAL QIREN BEST-EPOCH RESULTS")
print("=" * 80)

print(paper_df.round(4).to_string(index=False))

print("\n" + "=" * 80)
print("COMPLETE")
print("=" * 80)

print("Experiments:", len(df))
print("Checkpoints: Best + Last for every seed")
print("All results saved to:", BEST_ROOT)


Official repository: /content/QIREN_paper_recheck_repo/qinr
Commit: 56ab4010ee3fbb3371c9d5d5f5284a8c41d4e6f3
New output directory: /content/drive/MyDrive/QIREN_paper_recheck/best_epoch_runs
Original runs will NOT be overwritten.

START sound__hybridren__seed0 | epochs=300 | lr=0.03
  Training epoch 0/299 | logged MSE ×1e-3=344.2440
  Training epoch 100/299 | logged MSE ×1e-3=8.6960
  Training epoch 200/299 | logged MSE ×1e-3=7.9260
  Training epoch 299/299 | logged MSE ×1e-3=7.1420
  Training complete. Evaluating all 300 checkpoints...
  BEST epoch=298 | MSE=7.1417
  LAST epoch=299 | MSE=7.1559
  Best improvement=0.20%

START sound__hybridren__seed1 | epochs=300 | lr=0.03
  Training epoch 0/299 | logged MSE ×1e-3=134.7750
  Training epoch 100/299 | logged MSE ×1e-3=7.6390
  Training epoch 200/299 | logged MSE ×1e-3=7.0080
  Training epoch 299/299 | logged MSE ×1e-3=6.6300
  Training complete. Evaluating all 300 checkpoints...
  BEST epoch=270 | MSE=6.2782
  LAST epoch=299 | MSE=6.5686


In [ ]:
import pandas as pd
from pathlib import Path

root = Path("/content/drive/MyDrive/QIREN_paper_recheck/best_epoch_runs")

df = pd.read_csv(root / "best_vs_last_all_seeds.csv")
paper = pd.read_csv(root / "qiren_best_vs_paper.csv")

print("===== Training status =====")
print("Runs:", len(df))
print("Best weights:", len(list(root.glob("*__best.pth"))))
print("Last weights:", len(list(root.glob("*__last.pth"))))

print("\n===== 20 runs: best vs. last =====")
print(df.round(4).to_string(index=False))

print("\n===== Comparison with the paper =====")
print(paper.round(4).to_string(index=False))

===== Training status =====
Runs: 20
Best weights: 20
Last weights: 20

===== 20 runs: best vs. last =====
     task  seed  best_epoch  last_epoch  best_rep_mse  last_rep_mse  best_sr_mse  last_sr_mse
    sound     0         298         299        7.1417        7.1559          NaN          NaN
    sound     1         270         299        6.2782        6.5686          NaN          NaN
    sound     2         296         299        6.6364        7.0622          NaN          NaN
    sound     3         264         299        7.0419        8.6030          NaN          NaN
    sound     4         292         299        7.0157        7.1245          NaN          NaN
astronaut     0         581         599        4.9791        8.1836      26.0224      29.3520
astronaut     1         576         599        5.7375        8.1900      25.9971      28.4986
astronaut     2         593         599        5.5131        7.5468      26.0278      28.2700
astronaut     3         571         599        

In [ ]:
# ===== Restore the environment (run this cell first after every reconnect) =====
import os, sys, json, subprocess, importlib
from pathlib import Path
from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

OUT  = Path('/content/drive/MyDrive/QIREN_paper_recheck')
RUNS = OUT / 'runs'
ROOT = Path('/content/QIREN_paper_recheck_repo')
QINR = ROOT / 'qinr'

records = sorted(RUNS.glob('*.json'))
assert len(records) == 100, f'Expected 100 training records, found {len(records)}'
REPO_SHA = json.loads(records[0].read_text())['repo_sha']
assert all(json.loads(p.read_text())['repo_sha'] == REPO_SHA for p in records), 'Found different code versions'
print('Saved records:', len(records), '| weights:', len(list(RUNS.glob('*.pth'))), '| commit:', REPO_SHA[:8])

# 1. Restore the official repository to the commit used for training
if not (ROOT / '.git').is_dir():
    assert not ROOT.exists(), f'{ROOT} exists but is not a Git repository; delete it first'
    subprocess.run(['git', 'clone', '-q', 'https://github.com/GGorMM1/QIREN.git', str(ROOT)], check=True)
subprocess.run(['git', '-C', str(ROOT), 'checkout', '-q', REPO_SHA], check=True)
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'pennylane'], check=True)

# 2. Re-apply the patches (identical to cell 2 / cell 2b)
def checked_patch(relpath, replacements):
    p = QINR / relpath
    text = p.read_text(encoding='utf-8').replace('\r\n', '\n')
    if text.startswith('#QIREN-PAPER-RECHECK-PATCHED'):
        print('Already patched:', relpath); return
    for before, after in replacements:
        assert text.count(before) == 1, f'{relpath}: anchor found {text.count(before)} times: {before!r}'
        text = text.replace(before, after, 1)
    p.write_text('#QIREN-PAPER-RECHECK-PATCHED\n' + text, encoding='utf-8')
    print('Patched:', relpath)

checked_patch('modules.py', [
    ('import qiskit.providers.aer.noise as noise',
     'try:\n    import qiskit.providers.aer.noise as noise\nexcept ImportError:\n    noise = None'),
])
checked_patch('image/data_process.py', [
    ('def get_cameraman_tensor(sidelength):', "def get_cameraman_tensor(sidelength, name='coffee'):"),
    ('    img = Image.fromarray(skimage.color.rgb2gray(skimage.data.coffee()))\n',
     "    if name == 'camera':\n        img = Image.fromarray(skimage.data.camera())\n"
     "    elif name == 'astronaut':\n        img = Image.fromarray(skimage.color.rgb2gray(skimage.data.astronaut()))\n"
     "    elif name == 'coffee':\n        img = Image.fromarray(skimage.color.rgb2gray(skimage.data.coffee()))\n"
     "    else:\n        raise ValueError(f'Unknown image: {name}')\n"),
    ('    def __init__(self, sidelength):', "    def __init__(self, sidelength, name='coffee'):"),
    ('        img = get_cameraman_tensor(sidelength)', '        img = get_cameraman_tensor(sidelength, name)'),
])
checked_patch('train.py', [
    ("parser.add_argument('--use_noise', type=float, default=0)\n",
     "parser.add_argument('--use_noise', type=float, default=0)\n"
     "parser.add_argument('--image', type=str, choices=['coffee', 'camera', 'astronaut'], default='coffee')\n"
     "parser.add_argument('--ckpt_dir', type=str, default='./checkpoint')\n"
     "parser.add_argument('--seed', type=int, default=-1)\n"),
    ('args = parser.parse_args()\n',
     'args = parser.parse_args()\nif args.seed >= 0:\n    import random\n    random.seed(args.seed)\n'
     '    np.random.seed(args.seed)\n    torch.manual_seed(args.seed)\n'),
    ('path = "./checkpoint/" + str(epoch) + ".pth"',
     'os.makedirs(args.ckpt_dir, exist_ok=True); path = os.path.join(args.ckpt_dir, str(epoch) + ".pth")'),
    ('ImageFitting(args.img_size)', 'ImageFitting(args.img_size, args.image)'),
])
mp = QINR / 'modules.py'
s = mp.read_text(encoding='utf-8')
if 'qml.RZ(inputs[..., j], wires=j)' not in s:
    assert s.count('qml.RZ(inputs[j], wires=j)') == 1
    mp.write_text(s.replace('qml.RZ(inputs[j], wires=j)', 'qml.RZ(inputs[..., j], wires=j)', 1), encoding='utf-8')
    print('Quantum-layer batch indexing fixed')
subprocess.run([sys.executable, '-m', 'py_compile', 'train.py', 'modules.py', 'image/data_process.py'], cwd=QINR, check=True)

# 3. Restore variables and functions
import torch, pandas as pd
os.chdir(QINR)
if str(QINR) not in sys.path: sys.path.insert(0, str(QINR))
for m in ('modules', 'image.data_process', 'sound.data_process'):
    sys.modules.pop(m, None)
from modules import Hybridren, Siren, Siren_bn
from image.data_process import ImageFitting
from sound.data_process import AudioFile

SEEDS = (0, 1, 2, 3, 4)
PAPER_EPOCHS = {'sound': 300, 'coffee': 300, 'camera': 300, 'astronaut': 600}
EXPECTED = {'relu': (831, 841), 'tanh': (831, 841), 'relu+rff': (791, 791), 'siren': (691, 701), 'hybridren': (649, 657)}
MODELS = ('relu', 'tanh', 'relu+rff', 'siren', 'hybridren')

def make_model(model, in_f, omega=1):
    if model == 'hybridren':
        return Hybridren(in_features=in_f, out_features=1, hidden_features=8, hidden_layers=2, spectrum_layer=2, use_noise=0)
    if model == 'siren':
        return Siren(in_features=in_f, out_features=1, hidden_features=10, hidden_layers=6,
                     first_omega_0=omega, hidden_omega_0=omega)
    activ = {'relu': 'relu', 'tanh': 'tanh', 'relu+rff': 'relu'}[model]
    return Siren_bn(in_features=in_f, out_features=1, hidden_features=10, hidden_layers=6,
                    first_omega_0=1, hidden_omega_0=1, activ=activ, rff=(model == 'relu+rff'))

for model, exp in EXPECTED.items():     # parameter-count self-check
    assert sum(p.numel() for p in make_model(model, 1).parameters()) == exp[0], model
print('Environment restored ✅\n')

# ===== Baselines: "best epoch" representation MSE (read from existing records, no rerun) =====
rows = []
for jf in sorted(RUNS.glob('*.json')):
    r = json.loads(jf.read_text())
    L = [x * 1000 for x in r['losses']]
    rows.append(dict(task=r['task'], model=r['model'], seed=r['seed'],
                     best_rep=min(L), last_rep=L[-1]))
d = pd.DataFrame(rows)
print('Representation MSE x1e-3 (best of 5 seeds): best = lowest over all epochs, last = final epoch')
print(d.groupby(['task', 'model'])[['best_rep', 'last_rep']].min().round(3))

Mounted at /content/drive
Saved records: 100 | weights: 100 | commit: 56ab4010
Patched: modules.py
Patched: image/data_process.py
Patched: train.py
Quantum-layer batch indexing fixed
Environment restored ✅

Representation MSE x1e-3 (best of 5 seeds): best = lowest over all epochs, last = final epoch
                     best_rep  last_rep
task      model                        
astronaut hybridren     4.943     5.633
          relu          9.940    12.134
          relu+rff      6.392     6.984
          siren         2.088     2.454
          tanh         18.261    19.476
camera    hybridren     1.850     2.491
          relu          2.964     3.656
          relu+rff      2.902     3.359
          siren         0.553     0.556
          tanh          6.661     6.942
coffee    hybridren     1.767     1.879
          relu          4.850     5.137
          relu+rff      4.199     4.351
          siren         1.002     1.379
          tanh         18.413    18.536
sound     hybridren

In [ ]:
# ===== Baselines: super-resolution from the "best epoch" weights (60 retraining runs; seeds, learning rates and omega exactly as before) =====
import re, json, time, shutil, subprocess
import numpy as np, pandas as pd, torch
from pathlib import Path

BASE_OUT = OUT / 'baseline_best_epoch'; BASE_OUT.mkdir(exist_ok=True)
TMP = Path('/content/qiren_baseline_tmp'); TMP.mkdir(exist_ok=True)
BASELINES = ('relu', 'tanh', 'relu+rff', 'siren')
IMAGE_TASKS = ('astronaut', 'camera', 'coffee')
DATA32 = {t: ImageFitting(32, t)[0] for t in IMAGE_TASKS}
DATA64 = {t: ImageFitting(64, t)[0] for t in IMAGE_TASKS}

def eval_ckpt(model_name, omega, path, data):
    net = make_model(model_name, 2, omega=omega)
    net.load_state_dict(torch.load(path, map_location='cpu', weights_only=False)['model_state_dict'], strict=True)
    net.train()                     # as in the official trainer: BatchNorm uses the statistics of the current coordinates
    x, y = data
    with torch.no_grad():
        pred, _ = net(x.unsqueeze(0))
    return ((pred[0] - y) ** 2).mean().item() * 1000

for model in BASELINES:
    for task in IMAGE_TASKS:
        for seed in SEEDS:
            tag = f'{task}__{model}__seed{seed}'
            out_json = BASE_OUT / f'{tag}.json'
            if out_json.exists():
                continue
            old = json.loads((RUNS / f'{tag}.json').read_text())
            E = PAPER_EPOCHS[task]
            ck = TMP / tag
            shutil.rmtree(ck, ignore_errors=True); ck.mkdir(parents=True)
            cmd = list(old['command'])
            cmd[cmd.index('--ckpt_dir') + 1] = str(ck)      # all other arguments unchanged
            t0 = time.time()
            p = subprocess.run(cmd, cwd=QINR, capture_output=True, text=True)
            losses = [float(m[1]) for m in re.finditer(r'epoch\s+\d+,\s+Total loss\s+([-\deE+.]+)', p.stdout)]
            assert p.returncode == 0 and len(losses) == E, f'{tag} failed: rc={p.returncode}, epochs={len(losses)}\n{p.stdout[-1500:]}'
            same = bool(np.allclose(losses, old['losses'], rtol=1e-4))
            e = int(np.argmin(losses[1:])) + 1          # loss of epoch e = weights of checkpoint e-1
            best_ck = ck / f'{e - 1}.pth'
            rep = eval_ckpt(model, old['omega'], best_ck, DATA32[task])
            sr = eval_ckpt(model, old['omega'], best_ck, DATA64[task])
            if abs(rep - losses[e] * 1000) > 1e-3 * max(1.0, rep):
                print(f'  Warning: recomputed checkpoint {rep:.4f} differs from log {losses[e]*1000:.4f}')
            shutil.copy2(best_ck, BASE_OUT / f'{tag}__best.pth')
            shutil.rmtree(ck, ignore_errors=True)
            rec = dict(task=task, model=model, seed=seed, lr=old['lr'], omega=old['omega'], epochs=E,
                       best_epoch_logged=e, best_logged_mse=losses[e] * 1000, last_logged_mse=losses[-1] * 1000,
                       best_rep_mse=rep, best_sr_mse=sr, reproduced_original_run=same, seconds=round(time.time() - t0, 1))
            tmp = out_json.with_suffix('.tmp'); tmp.write_text(json.dumps(rec, indent=2)); tmp.replace(out_json)
            print(f'{tag}: best epoch {e}, rep {rep:.3f}, SR {sr:.2f}, same_as_original={same}, {rec["seconds"]:.0f}s', flush=True)

# ---------- Summary: compare with QIREN (best epoch, already available) and the paper ----------
PAPER_REP = {'astronaut': {'relu': 9.9, 'tanh': 20.7, 'relu+rff': 5.1, 'siren': 9.0, 'hybridren': 4.0},
             'camera':    {'relu': 2.7, 'tanh': 5.8,  'relu+rff': 1.9, 'siren': 1.5, 'hybridren': 1.1},
             'coffee':    {'relu': 4.2, 'tanh': 14.8, 'relu+rff': 4.9, 'siren': 2.3, 'hybridren': 1.5}}
PAPER_SR  = {'astronaut': {'relu': 33.8, 'tanh': 47.8, 'relu+rff': 39.9, 'siren': 77.0, 'hybridren': 24.3},
             'camera':    {'relu': 11.3, 'tanh': 15.2, 'relu+rff': 13.3, 'siren': 26.3, 'hybridren': 7.9},
             'coffee':    {'relu': 13.5, 'tanh': 26.7, 'relu+rff': 23.9, 'siren': 15.7, 'hybridren': 9.4}}
b = pd.DataFrame([json.loads(f.read_text()) for f in sorted(BASE_OUT.glob('*.json'))])
print('\nShare of baseline retraining runs identical to the original run:', f"{b.reproduced_original_run.mean()*100:.0f}%  ({len(b)} runs)")
q = pd.read_csv(OUT / 'best_epoch_runs' / 'best_vs_last_all_seeds.csv'); q['model'] = 'hybridren'
cols = ['task', 'model', 'seed', 'best_rep_mse', 'best_sr_mse']
allr = pd.concat([b[cols], q[q.task != 'sound'][cols]], ignore_index=True)
rows = []
for task in IMAGE_TASKS:
    for model in ('relu', 'tanh', 'relu+rff', 'siren', 'hybridren'):
        g = allr[(allr.task == task) & (allr.model == model)]
        if g.empty: continue
        i = g.best_rep_mse.idxmin()
        rows.append(dict(task=task, model=model, n=len(g), rep=g.best_rep_mse.min(), rep_paper=PAPER_REP[task][model],
                         sr_same_seed=g.loc[i, 'best_sr_mse'], sr_best_any=g.best_sr_mse.min(), sr_paper=PAPER_SR[task][model]))
summary = pd.DataFrame(rows)
summary.to_csv(BASE_OUT / 'summary_best_epoch_all_models.csv', index=False)
print('\nMSE x1e-3 (all models use the "best epoch"; sr_same_seed = super-resolution of the seed that is best on representation)')
print(summary.round(2).to_string(index=False))

/usr/local/lib/python3.13/dist-packages/torch/functional.py:504: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/native/TensorShape.cpp:4221.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


astronaut__relu__seed0: best epoch 589, rep 12.410, SR 37.58, same_as_original=False, 8s
astronaut__relu__seed1: best epoch 592, rep 15.216, SR 46.03, same_as_original=False, 6s
astronaut__relu__seed2: best epoch 583, rep 13.021, SR 37.89, same_as_original=False, 6s
astronaut__relu__seed3: best epoch 564, rep 13.963, SR 39.29, same_as_original=False, 6s
astronaut__relu__seed4: best epoch 586, rep 23.145, SR 54.39, same_as_original=False, 6s
camera__relu__seed0: best epoch 298, rep 5.025, SR 14.64, same_as_original=False, 5s
camera__relu__seed1: best epoch 285, rep 3.626, SR 12.23, same_as_original=False, 5s
camera__relu__seed2: best epoch 288, rep 4.027, SR 12.18, same_as_original=False, 5s
camera__relu__seed3: best epoch 295, rep 2.913, SR 11.37, same_as_original=False, 5s
camera__relu__seed4: best epoch 284, rep 4.690, SR 13.14, same_as_original=False, 5s
coffee__relu__seed0: best epoch 279, rep 5.469, SR 14.40, same_as_original=False, 5s
coffee__relu__seed1: best epoch 293, rep 5.73

In [ ]:
# ===== SIREN: official default hyperparameters (omega=1, lr=0.03) vs. tuned vs. QIREN =====
# Prerequisite: run the "restore environment" cell first (needs OUT, RUNS, QINR, SEEDS, make_model, ImageFitting, AudioFile)
import re, json, shutil, subprocess
import numpy as np, pandas as pd, torch
from pathlib import Path

TASKS = ('sound', 'astronaut', 'camera', 'coffee'); IMG = TASKS[1:]
OMEGAS, LRS = (1, 10, 30), (1e-4, 1e-3, 1e-2, 3e-2)
PAPER_REP = {'sound': {'siren': 5.5, 'hybridren': 5.5}, 'astronaut': {'siren': 9.0, 'hybridren': 4.0},
             'camera': {'siren': 1.5, 'hybridren': 1.1}, 'coffee': {'siren': 2.3, 'hybridren': 1.5}}
PAPER_SR = {'astronaut': {'siren': 77.0, 'hybridren': 24.3}, 'camera': {'siren': 26.3, 'hybridren': 7.9},
            'coffee': {'siren': 15.7, 'hybridren': 9.4}}

# ---------- Part A: existing tuning grid (seed 123, final-epoch loss x1e-3, no rerun) ----------
grid = {}
for f in (OUT / 'siren_tune').glob('omega*__lr*__*__siren__seed123.json'):
    m = re.match(r'omega(\d+)__lr([0-9.e-]+)__(\w+?)__siren__seed123', f.name)
    if m:
        grid[(m[3], int(m[1]), float(m[2]))] = json.loads(f.read_text())['last_pre_update_loss'] * 1000
for task in TASKS:
    t = pd.DataFrame([[grid.get((task, w, lr), np.nan) for lr in LRS] for w in OMEGAS],
                     index=[f'omega={w}' for w in OMEGAS], columns=[f'lr={lr:g}' for lr in LRS])
    print(f'\n{task}   SIREN grid (official default = omega=1, lr=0.03)')
    print(t.round(2).to_string())

# ---------- Part B: official default SIREN, 5 seeds x 4 tasks ----------
DEF = OUT / 'siren_repo_default'; DEF.mkdir(exist_ok=True)
TMP = Path('/content/qiren_siren_default_tmp'); TMP.mkdir(exist_ok=True)
DATA32 = {'sound': AudioFile(str(QINR / 'sound' / 'gt_bach.wav'))[0], **{t: ImageFitting(32, t)[0] for t in IMG}}
DATA64 = {t: ImageFitting(64, t)[0] for t in IMG}

def eval_ckpt(task, path, data):
    net = make_model('siren', 1 if task == 'sound' else 2, omega=1)
    net.load_state_dict(torch.load(path, map_location='cpu', weights_only=False)['model_state_dict'], strict=True)
    net.train()                                  # as in the official trainer
    x, y = data
    with torch.no_grad():
        pred, _ = net(x.unsqueeze(0))
    return ((pred[0] - y) ** 2).mean().item() * 1000

def setarg(cmd, name, val):
    cmd[cmd.index(name) + 1] = str(val)

for task in TASKS:
    for seed in SEEDS:
        tag = f'{task}__siren_default__seed{seed}'
        out = DEF / f'{tag}.json'
        if out.exists():
            continue
        old = json.loads((RUNS / f'{task}__siren__seed{seed}.json').read_text())
        cmd = list(old['command'])
        ck = TMP / tag; shutil.rmtree(ck, ignore_errors=True); ck.mkdir(parents=True)
        setarg(cmd, '--ckpt_dir', ck); setarg(cmd, '--learning_rate', 0.03)
        setarg(cmd, '--first_omega_0', 1); setarg(cmd, '--hidden_omega_0', 1)   # official defaults
        p = subprocess.run(cmd, cwd=QINR, capture_output=True, text=True)
        losses = [float(m[1]) for m in re.finditer(r'epoch\s+\d+,\s+Total loss\s+([-\deE+.naif]+)', p.stdout)]
        E = len(old['losses'])
        assert p.returncode == 0 and len(losses) == E, f'{tag} failed: rc={p.returncode}, epochs={len(losses)}\n{p.stdout[-1500:]}'
        e = int(np.nanargmin(losses[1:])) + 1             # loss of epoch e = checkpoint e-1
        best_ck = ck / f'{e - 1}.pth'
        rep = eval_ckpt(task, best_ck, DATA32[task])
        sr = None if task == 'sound' else eval_ckpt(task, best_ck, DATA64[task])
        if abs(rep - losses[e] * 1000) > 1e-3 * max(1.0, rep):
            print(f'  Warning: recomputed checkpoint {rep:.4f} differs from log {losses[e]*1000:.4f}')
        shutil.rmtree(ck, ignore_errors=True)
        out.write_text(json.dumps(dict(task=task, seed=seed, lr=0.03, omega=1, best_epoch=e,
                                       best_logged=losses[e] * 1000, last_logged=losses[-1] * 1000,
                                       best_rep=rep, best_sr=sr)))
        print(f'{tag}: best epoch {e}, rep {losses[e]*1000:.3f}, last {losses[-1]*1000:.3f}', flush=True)

# ---------- Part C: summary ----------
def from_runs(task, model):                       # existing 100 runs: "best epoch" over 5 seeds
    return min(min(json.loads((RUNS / f'{task}__{model}__seed{s}.json').read_text())['losses'])
               for s in SEEDS) * 1000

d = pd.DataFrame([json.loads(f.read_text()) for f in DEF.glob('*.json')])
rows = []
for task in TASKS:
    g = d[d.task == task]
    rows.append(dict(task=task, SIREN_default_best=g.best_logged.min(), SIREN_default_last=g.last_logged.min(),
                     SIREN_tuned=from_runs(task, 'siren'), QIREN=from_runs(task, 'hybridren'),
                     paper_SIREN=PAPER_REP[task]['siren'], paper_QIREN=PAPER_REP[task]['hybridren']))
print('\nRepresentation MSE x1e-3 (best of 5 seeds; best = lowest over all epochs, last = final epoch)')
print(pd.DataFrame(rows).round(2).to_string(index=False))

sp = OUT / 'baseline_best_epoch' / 'summary_best_epoch_all_models.csv'
if sp.exists():
    s = pd.read_csv(sp); sr_rows = []
    for task in IMG:
        g = d[d.task == task]; i = g.best_logged.idxmin()         # keep the seed that is best on representation
        sr_rows.append(dict(task=task, SIREN_default=g.loc[i, 'best_sr'],
                            SIREN_tuned=s[(s.task == task) & (s.model == 'siren')].sr_same_seed.iloc[0],
                            QIREN=s[(s.task == task) & (s.model == 'hybridren')].sr_same_seed.iloc[0],
                            paper_SIREN=PAPER_SR[task]['siren'], paper_QIREN=PAPER_SR[task]['hybridren']))
    print('\nSuper-resolution MSE x1e-3 (same seed)')
    print(pd.DataFrame(sr_rows).round(2).to_string(index=False))
else:
    print('\n(baseline_best_epoch summary not found; skipping the super-resolution comparison)')


sound   SIREN grid (official default = omega=1, lr=0.03)
          lr=0.0001  lr=0.001  lr=0.01  lr=0.03
omega=1       32.52     20.95    17.40    12.08
omega=10      21.93      5.98   128.75   128.65
omega=30      63.52      5.23   129.95   129.02

astronaut   SIREN grid (official default = omega=1, lr=0.03)
          lr=0.0001  lr=0.001  lr=0.01  lr=0.03
omega=1      162.75    110.44    38.08    15.86
omega=10      35.33      4.82     3.59   256.63
omega=30      69.32      6.10   256.02   256.46

camera   SIREN grid (official default = omega=1, lr=0.03)
          lr=0.0001  lr=0.001  lr=0.01  lr=0.03
omega=1      105.03     61.93    24.16     8.59
omega=10      40.11      4.72     0.82   288.55
omega=30      87.29      1.59   289.59   290.47

coffee   SIREN grid (official default = omega=1, lr=0.03)
          lr=0.0001  lr=0.001  lr=0.01  lr=0.03
omega=1      109.05     75.00    27.65    11.39
omega=10      67.81      4.23     2.78   198.94
omega=30     128.51      6.67   199.82   2

In [ ]:
import json, glob, pandas as pd
from pathlib import Path

LR_GRID = [0.003, 0.01, 0.03, 0.1]
TASKS_ = ['astronaut', 'camera', 'coffee']     # add other tasks (e.g. sound) here if needed
SEED_ = 0
LR_OUT = OUT / 'qiren_lr_search'
LR_OUT.mkdir(exist_ok=True)

for lr in LR_GRID:
    for task in TASKS_:
        tag = f'lr{lr}_'
        if list(LR_OUT.glob(f'{tag}*{task}*seed{SEED_}*.json')):
            continue                                   # already done; resumable after a disconnect
        run_one(task, 'hybridren', SEED_, lr=lr, output_dir=LR_OUT, prefix=tag)

rows = []
for lr in LR_GRID:
    for task in TASKS_:
        fs = list(LR_OUT.glob(f'lr{lr}_*{task}*seed{SEED_}*.json'))
        if not fs:
            continue
        d = json.loads(fs[0].read_text())
        if rows == []:
            print('JSON fields:', list(d.keys()))        # check these if the metric cannot be read below
        loss = d.get('losses') or d.get('loss') or d.get('train_loss')
        best = min(loss) if isinstance(loss, list) else d.get('best_loss', d.get('final_loss', float('nan')))
        rows.append(dict(task=task, lr=lr, best_mse_e3=round(best * 1e3, 3),
                         t_s=round(d.get('duration_s', float('nan')), 1)))

df = pd.DataFrame(rows)
print(df.pivot(index='task', columns='lr', values='best_mse_e3'))
print('\nBest lr per task:')
print(df.loc[df.groupby('task').best_mse_e3.idxmin()][['task', 'lr', 'best_mse_e3']].to_string(index=False))
print('\nTotal time (min):', round(df.t_s.sum() / 60, 1))
df.to_csv(LR_OUT / 'qiren_lr_search.csv', index=False)

NameError: name 'run_one' is not defined

In [ ]:
import sys, os, subprocess, shutil, json, re, time, pathlib
import pandas as pd

TEMP = pathlib.Path('/content/qiren_tmp_ckpt'); TEMP.mkdir(exist_ok=True)
LR_OUT = OUT / 'qiren_lr_search'; LR_OUT.mkdir(exist_ok=True)

TASK_ARGS = {
    'sound':     ('--type','sound','--in_features','1'),
    'coffee':    ('--type','image','--in_features','2','--image','coffee'),
    'camera':    ('--type','image','--in_features','2','--image','camera'),
    'astronaut': ('--type','image','--in_features','2','--image','astronaut'),
}

def run_qiren(task, seed, lr):
    E = PAPER_EPOCHS[task]
    tag = f'lr{lr}__{task}__hybridren__seed{seed}'
    jf, wf = LR_OUT / f'{tag}.json', LR_OUT / f'{tag}.pth'
    if jf.exists() and wf.exists():
        print('SKIP', tag); return json.loads(jf.read_text())
    ck = TEMP / tag
    if ck.exists(): shutil.rmtree(ck)
    ck.mkdir(parents=True)
    cmd = [sys.executable, '-u', 'train.py', *TASK_ARGS[task],
           '--model', 'hybridren', '--hidden_features', '8', '--hidden_layers', '2',
           '--spectrum_layer', '2', '--use_noise', '0',
           '--learning_rate', str(lr), '--seed', str(seed),
           '--epochs', str(E), '--ckpt_dir', str(ck)]
    print('START', tag, flush=True)
    t0 = time.time(); losses = []; tail = []
    with open(LR_OUT / f'{tag}.log', 'w', encoding='utf-8') as log:
        p = subprocess.Popen(cmd, cwd=QINR, stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            log.write(line); tail = (tail + [line.rstrip()])[-30:]
            if m := re.search(r'epoch\s+(\d+),\s+Total loss\s+([-\deE+.]+)', line):
                losses.append(float(m.group(2)))
        rc = p.wait()
    if rc != 0 or len(losses) != E:
        raise RuntimeError(f'FAILED {tag}: rc={rc}, epochs={len(losses)}/{E}\n' + '\n'.join(tail))
    shutil.copy2(ck / f'{E-1}.pth', wf)
    rec = dict(task=task, seed=seed, lr=lr, epochs=E, duration_s=round(time.time()-t0, 1),
               losses=losses, best_loss=min(losses), best_epoch=int(min(range(E), key=losses.__getitem__)))
    jf.write_text(json.dumps(rec))
    shutil.rmtree(ck, ignore_errors=True)
    print('DONE', tag, 'best MSE×1e-3 =', round(rec['best_loss']*1e3, 4), f"({rec['duration_s']}s)", flush=True)
    return rec

LR_GRID, TASKS_, SEED_ = [0.003, 0.01, 0.03, 0.1], ['sound','coffee','camera','astronaut'], 0
rows = [run_qiren(t, SEED_, lr) for lr in LR_GRID for t in TASKS_]

df = pd.DataFrame([dict(task=r['task'], lr=r['lr'], best_mse_e3=round(r['best_loss']*1e3, 4),
                        best_epoch=r['best_epoch'], t_s=r['duration_s']) for r in rows])
print('\nbest-epoch MSE ×1e-3 (lower is better)')
print(df.pivot(index='task', columns='lr', values='best_mse_e3'))
print('\nBest lr per task:')
print(df.loc[df.groupby('task').best_mse_e3.idxmin()][['task','lr','best_mse_e3']].to_string(index=False))
print('\nTotal time (min):', round(df.t_s.sum()/60, 1))
df.to_csv(LR_OUT / 'qiren_lr_search.csv', index=False)

START lr0.003__sound__hybridren__seed0
DONE lr0.003__sound__hybridren__seed0 best MSE×1e-3 = 8.219 (121.1s)
START lr0.003__coffee__hybridren__seed0


KeyboardInterrupt: 

In [ ]:
import sys, os, subprocess, shutil, json, re, time, math, pathlib
import pandas as pd

TEMP = pathlib.Path('/content/qiren_tmp_ckpt'); TEMP.mkdir(exist_ok=True)
SG = OUT / 'siren_grid_vs_paper'; SG.mkdir(exist_ok=True)
IMG = {'coffee':('--image','coffee'), 'camera':('--image','camera'), 'astronaut':('--image','astronaut')}
PAPER_REP = {'astronaut': 9.0, 'camera': 1.5, 'coffee': 2.3}      # paper's SIREN representation MSE ×1e-3
OMEGAS, LRS, SEED_ = [1, 3, 10, 30], [1e-3, 3e-3, 1e-2, 3e-2], 0

def run_siren(task, omega, lr, seed=SEED_):
    E = PAPER_EPOCHS[task]
    tag = f'sg__{task}__om{omega}__lr{lr}__seed{seed}'
    jf = SG / f'{tag}.json'
    if jf.exists():
        return json.loads(jf.read_text())
    ck = TEMP / tag
    if ck.exists(): shutil.rmtree(ck)
    ck.mkdir(parents=True)
    cmd = [sys.executable, '-u', 'train.py', '--type', 'image', '--in_features', '2', *IMG[task],
           '--model', 'siren', '--hidden_features', '10', '--hidden_layers', '6',
           '--first_omega_0', str(omega), '--hidden_omega_0', str(omega),
           '--learning_rate', str(lr), '--seed', str(seed), '--epochs', str(E), '--ckpt_dir', str(ck)]
    t0 = time.time(); losses = []; tail = []
    p = subprocess.Popen(cmd, cwd=QINR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        tail = (tail + [line.rstrip()])[-30:]
        if m := re.search(r'epoch\s+(\d+),\s+Total loss\s+([-\deE+.]+)', line):
            losses.append(float(m.group(2)))
    rc = p.wait()
    if rc != 0 or len(losses) != E:
        raise RuntimeError(f'FAILED {tag}\n' + '\n'.join(tail))
    rec = dict(task=task, omega=omega, lr=lr, seed=seed, best_loss=min(losses), last_loss=losses[-1],
               duration_s=round(time.time()-t0, 1))
    jf.write_text(json.dumps(rec)); shutil.rmtree(ck, ignore_errors=True)
    print('DONE', tag, round(rec['best_loss']*1e3, 3), flush=True)
    return rec

rows = [run_siren(t, om, lr) for om in OMEGAS for lr in LRS for t in IMG]
df = pd.DataFrame(rows)
df['mse_e3'] = (df.best_loss * 1e3).round(3)
df['paper'] = df.task.map(PAPER_REP)
df['abs_log_ratio'] = (df.mse_e3 / df.paper).apply(lambda r: abs(math.log(r)))

print('Representation best MSE ×1e-3 (paper: astronaut 9.0 / camera 1.5 / coffee 2.3)')
print(df.pivot_table(index=['omega','lr'], columns='task', values='mse_e3'))
score = df.groupby(['omega','lr']).abs_log_ratio.mean().sort_values()
print('\n5 settings closest to the paper (mean |ln(ours/paper)| over 3 tasks, lower is closer):')
print(score.head(5).round(3))
df.to_csv(SG / 'siren_grid_vs_paper.csv', index=False)

DONE sg__coffee__om1__lr0.001__seed0 73.208
DONE sg__camera__om1__lr0.001__seed0 49.166
DONE sg__astronaut__om1__lr0.001__seed0 91.727
DONE sg__coffee__om1__lr0.003__seed0 70.974
DONE sg__camera__om1__lr0.003__seed0 23.536
DONE sg__astronaut__om1__lr0.003__seed0 38.582
DONE sg__coffee__om1__lr0.01__seed0 30.696
DONE sg__camera__om1__lr0.01__seed0 17.141
DONE sg__astronaut__om1__lr0.01__seed0 23.008
DONE sg__coffee__om1__lr0.03__seed0 22.261
DONE sg__camera__om1__lr0.03__seed0 10.108
DONE sg__astronaut__om1__lr0.03__seed0 17.884
DONE sg__coffee__om3__lr0.001__seed0 26.333
DONE sg__camera__om3__lr0.001__seed0 8.753
DONE sg__astronaut__om3__lr0.001__seed0 19.871
DONE sg__coffee__om3__lr0.003__seed0 7.646
DONE sg__camera__om3__lr0.003__seed0 3.927
DONE sg__astronaut__om3__lr0.003__seed0 8.337
DONE sg__coffee__om3__lr0.01__seed0 2.739
DONE sg__camera__om3__lr0.01__seed0 2.648
DONE sg__astronaut__om3__lr0.01__seed0 4.347
DONE sg__coffee__om3__lr0.03__seed0 3.819
DONE sg__camera__om3__lr0.03_

In [ ]:
import sys, os, subprocess, shutil, json, re, time, math, pathlib
import numpy as np, pandas as pd, torch
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ====== Settings ======
SEEDS = [0, 1, 2]                       # use [0] for a faster run
TASKS = ['astronaut', 'camera', 'coffee']
MODELS = ['relu', 'tanh', 'relu+rff', 'siren', 'hybridren']          # hybridren = QIREN
LRS = [1e-3, 3e-3, 1e-2, 3e-2]
OMEGAS = [1, 3, 10, 30]                 # SIREN only
QIREN_LRS = [3e-3, 1e-2, 3e-2, 1e-1]

PM = OUT / 'paper_match'; PM.mkdir(exist_ok=True)
(PM / 'ckpt').mkdir(exist_ok=True); (PM / 'sr').mkdir(exist_ok=True)
TEMP = pathlib.Path('/content/qiren_tmp_ckpt'); TEMP.mkdir(exist_ok=True)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'

# Values from the paper's table (MSE ×1e-3)
PAPER_REP = {'astronaut': dict(relu=9.9, tanh=20.7, **{'relu+rff': 5.1}, siren=9.0, hybridren=4.0),
             'camera':    dict(relu=2.7, tanh=5.8,  **{'relu+rff': 1.9}, siren=1.5, hybridren=1.1),
             'coffee':    dict(relu=4.2, tanh=14.8, **{'relu+rff': 4.9}, siren=2.3, hybridren=1.5)}
PAPER_SR  = {'astronaut': dict(relu=33.8, tanh=47.8, **{'relu+rff': 39.9}, siren=77.0, hybridren=24.3),
             'camera':    dict(relu=11.3, tanh=15.2, **{'relu+rff': 13.3}, siren=26.3, hybridren=7.9),
             'coffee':    dict(relu=13.5, tanh=26.7, **{'relu+rff': 23.9}, siren=15.7, hybridren=9.4)}

if 'ImageFitting' not in globals():
    sys.path.insert(0, str(QINR))
    from image.data_process import ImageFitting

IMG = {t: ('--type', 'image', '--in_features', '2', '--image', t) for t in TASKS}

def grid(model):
    if model == 'siren':     return [dict(lr=l, omega=o) for o in OMEGAS for l in LRS]
    if model == 'hybridren': return [dict(lr=l, omega=1) for l in QIREN_LRS]
    return [dict(lr=l, omega=1) for l in LRS]

@torch.no_grad()
def evaluate(model_name, omega, ckpt, task):
    m = make_model(model_name, 2, omega).to(dev)
    sd = torch.load(ckpt, map_location=dev)
    m.load_state_dict(sd.get('model_state_dict', sd) if isinstance(sd, dict) else sd)
    m.train()                                      # as in the original code: BatchNorm uses batch statistics
    res = {}
    for side in (32, 64):
        coords, pix = ImageFitting(side, task)[0]
        out = m(coords.unsqueeze(0).to(dev))
        out = out[0] if isinstance(out, tuple) else out
        out = out.reshape(-1, 1).float().cpu()
        res[side] = (float(((out - pix.reshape(-1, 1).float()) ** 2).mean()), out.numpy().reshape(side, side))
    return res

def run(task, model, omega, lr, seed):
    E = PAPER_EPOCHS[task]
    tag = f'{task}__{model}__om{omega}__lr{lr}__seed{seed}'
    jf = PM / f'{tag}.json'
    if jf.exists():
        return json.loads(jf.read_text())
    hf, hl = (8, 2) if model == 'hybridren' else (10, 6)
    args = ['--model', model, '--hidden_features', str(hf), '--hidden_layers', str(hl),
            '--learning_rate', str(lr), '--seed', str(seed)]
    if model == 'hybridren': args += ['--spectrum_layer', '2', '--use_noise', '0']
    if model == 'siren':     args += ['--first_omega_0', str(omega), '--hidden_omega_0', str(omega)]
    ck = TEMP / tag
    if ck.exists(): shutil.rmtree(ck)
    ck.mkdir(parents=True)
    cmd = [sys.executable, '-u', 'train.py', *IMG[task], *args, '--epochs', str(E), '--ckpt_dir', str(ck)]
    t0 = time.time(); losses = []; tail = []
    p = subprocess.Popen(cmd, cwd=QINR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        tail = (tail + [line.rstrip()])[-30:]
        if mm := re.search(r'epoch\s+(\d+),\s+Total loss\s+([-\deE+.nainf]+)', line):
            losses.append(float(mm.group(2)))
    rc = p.wait()
    if rc != 0 or len(losses) != E:
        raise RuntimeError(f'FAILED {tag}\n' + '\n'.join(tail))
    be = int(np.nanargmin(losses))
    src = ck / f'{max(be - 1, 0)}.pth'                 # the loss of epoch e belongs to the weights after step e-1
    if not src.is_file(): src = ck / f'{E - 1}.pth'
    bw = PM / 'ckpt' / f'{tag}.pth'; shutil.copy2(src, bw)
    r = evaluate(model, omega, bw, task)
    np.save(PM / 'sr' / f'{tag}.npy', r[64][1])
    rec = dict(task=task, model=model, omega=omega, lr=lr, seed=seed, best_epoch=be,
               rep=r[32][0] * 1e3, sr=r[64][0] * 1e3, secs=round(time.time() - t0, 1))
    jf.write_text(json.dumps(rec)); shutil.rmtree(ck, ignore_errors=True)
    return rec

# ====== Run everything (resumable after a disconnect) ======
recs, total = [], sum(len(grid(m)) for m in MODELS) * len(TASKS) * len(SEEDS)
for model in MODELS:
    for cfg in grid(model):
        for task in TASKS:
            for seed in SEEDS:
                recs.append(run(task, model, cfg['omega'], cfg['lr'], seed))
        print(f'{len(recs)}/{total}  {model} lr={cfg["lr"]} omega={cfg["omega"]} done', flush=True)

df = pd.DataFrame(recs)
df['paper_rep'] = [PAPER_REP[t][m] for t, m in zip(df.task, df.model)]
df['paper_sr'] = [PAPER_SR[t][m] for t, m in zip(df.task, df.model)]

# ====== Pick, per model, the hyperparameters closest to the paper (one setting shared by the 3 tasks) ======
agg = df.groupby(['model', 'omega', 'lr', 'task'], as_index=False).agg(
    rep=('rep', 'mean'), sr=('sr', 'mean'), paper_rep=('paper_rep', 'first'), paper_sr=('paper_sr', 'first'))
def dist(a, b):
    return abs(math.log(a / b)) if np.isfinite(a) and a > 0 else 99.0
agg['d'] = [(dist(r, pr) + dist(s, ps)) / 2 for r, pr, s, ps in zip(agg.rep, agg.paper_rep, agg.sr, agg.paper_sr)]
score = agg.groupby(['model', 'omega', 'lr']).d.mean().reset_index()
best = score.loc[score.groupby('model').d.idxmin()].set_index('model')
print('\nHyperparameters closest to the paper per model (lower d is closer; selected by the paper\'s numbers, not by independent tuning)')
print(best.round(3))

rows = []
for model in MODELS:
    om, lr = best.loc[model, 'omega'], best.loc[model, 'lr']
    for task in TASKS:
        s = df[(df.model == model) & (df.omega == om) & (df.lr == lr) & (df.task == task)]
        rows.append(dict(task=task, model=model, lr=lr, omega=om,
                         rep=f'{s.rep.mean():.2f}±{s.rep.std(ddof=0):.2f}', rep_paper=PAPER_REP[task][model],
                         sr=f'{s.sr.mean():.2f}±{s.sr.std(ddof=0):.2f}', sr_paper=PAPER_SR[task][model],
                         rep_ratio=round(s.rep.mean() / PAPER_REP[task][model], 2),
                         sr_ratio=round(s.sr.mean() / PAPER_SR[task][model], 2)))
final = pd.DataFrame(rows)
pd.set_option('display.width', 220)
print('\nFinal table: MSE ×1e-3, ours (mean ± std, %d seeds) vs. paper' % len(SEEDS))
print(final.to_string(index=False))
final.to_csv(PM / 'final_vs_paper.csv', index=False); df.to_csv(PM / 'all_runs.csv', index=False)

# ====== Figure: per task, ground truth + 64×64 super-resolution of the 5 models (seed that is best on representation) ======
fig, ax = plt.subplots(len(TASKS), 1 + len(MODELS), figsize=(2.6 * (1 + len(MODELS)), 2.7 * len(TASKS)))
for i, task in enumerate(TASKS):
    gt = ImageFitting(64, task)[0][1].reshape(64, 64).numpy()
    ax[i, 0].imshow(gt, cmap='gray', vmin=-1, vmax=1); ax[i, 0].set_title(f'{task}\nground truth 64×64', fontsize=9)
    for j, model in enumerate(MODELS, 1):
        om, lr = best.loc[model, 'omega'], best.loc[model, 'lr']
        s = df[(df.model == model) & (df.omega == om) & (df.lr == lr) & (df.task == task)]
        r = s.loc[s.rep.idxmin()]
        img = np.load(PM / 'sr' / f'{task}__{model}__om{om}__lr{lr}__seed{int(r.seed)}.npy')
        ax[i, j].imshow(img, cmap='gray', vmin=-1, vmax=1)
        ax[i, j].set_title(f"{'QIREN' if model == 'hybridren' else model}\nSR MSE {r.sr:.1f}e-3 (paper {PAPER_SR[task][model]})", fontsize=8)
for a in ax.ravel(): a.axis('off')
plt.tight_layout(); plt.savefig(PM / 'sr_reconstruction_vs_paper.png', dpi=150)

# Paper table vs. ours: bar chart
fig2, ax2 = plt.subplots(1, 2, figsize=(13, 4))
x = np.arange(len(TASKS) * len(MODELS)); lab = [f'{t[:4]}-{m[:5]}' for t in TASKS for m in MODELS]
for k, (col, pcol, name) in enumerate([('rep', 'rep_paper', 'Representation MSE ×1e-3'), ('sr', 'sr_paper', 'Super-resolution MSE ×1e-3')]):
    ours = [float(v.split('±')[0]) for v in final[col]]
    ax2[k].bar(x - 0.2, ours, 0.4, label='ours'); ax2[k].bar(x + 0.2, final[pcol], 0.4, label='paper')
    ax2[k].set_yscale('log'); ax2[k].set_xticks(x); ax2[k].set_xticklabels(lab, rotation=70, fontsize=7); ax2[k].set_title(name); ax2[k].legend()
plt.tight_layout(); plt.savefig(PM / 'table_vs_paper.png', dpi=150)
print('\nSaved to', PM)

9/288  relu lr=0.001 omega=1 done
18/288  relu lr=0.003 omega=1 done
27/288  relu lr=0.01 omega=1 done
36/288  relu lr=0.03 omega=1 done
45/288  tanh lr=0.001 omega=1 done
54/288  tanh lr=0.003 omega=1 done
63/288  tanh lr=0.01 omega=1 done
72/288  tanh lr=0.03 omega=1 done
81/288  relu+rff lr=0.001 omega=1 done
90/288  relu+rff lr=0.003 omega=1 done
99/288  relu+rff lr=0.01 omega=1 done
108/288  relu+rff lr=0.03 omega=1 done
117/288  siren lr=0.001 omega=1 done
126/288  siren lr=0.003 omega=1 done
135/288  siren lr=0.01 omega=1 done
144/288  siren lr=0.03 omega=1 done
153/288  siren lr=0.001 omega=3 done
162/288  siren lr=0.003 omega=3 done
171/288  siren lr=0.01 omega=3 done
180/288  siren lr=0.03 omega=3 done
189/288  siren lr=0.001 omega=10 done
198/288  siren lr=0.003 omega=10 done
207/288  siren lr=0.01 omega=10 done
216/288  siren lr=0.03 omega=10 done
225/288  siren lr=0.001 omega=30 done
234/288  siren lr=0.003 omega=30 done
243/288  siren lr=0.01 omega=30 done
252/288  siren 

In [ ]:
import numpy as np
import sys
import subprocess

version = np.__version__

subprocess.check_call([
    sys.executable, "-m", "pip",
    "install", "--force-reinstall",
    "--no-cache-dir", "--no-deps",
    f"numpy=={version}"
])

0